# VEP-nAChR2 — Experiment Results Log

Master results notebook tracking all experiment runs chronologically.
Each run documents: configuration, model comparison, ablation results, and key takeaways.

---
## Run I: Baseline — All Models + CatBoost Ablation
**Date:** 2026-08-03  
**Config:** 797 variants, 66 features (7 groups), 9 models, 5×3 nested gene-level CV, 20 Optuna trials  
**Imbalance:** cost_sensitive for majority, ROS for KNN/MLP/GaussianNB  
**GPU:** No (CPU-only — GPU slower on 797×66 dataset)

### Configuration

| Parameter | Value |
|-----------|-------|
| Dataset | 797 substitution variants (human nAChR) |
| Classes | 3: LOF (218), No net effect (193), GOF (386) |
| Features | 66 across 7 extractor groups |
| CV | Nested 5-fold StratifiedGroupKFold (gene-level) |
| Inner CV | 5-fold StratifiedKFold for HP tuning |
| Optuna | 20 trials, TPE sampler, MedianPruner, macro F1 objective |
| Seeds | 42, 123, 456 (3 seeds for comparison) |
| Feature groups | physicochemical (24), substitution (3), positional (20), structural_core (7), structural_nachr (7), conformational (5), embeddings (0) |
| PDB structures | 9DMG (muscle), 7EKT (α7 closed), 7KOX (α7 open), 6CNJ (α4β2), 6PV7 (α3β4), AF-Q9UGM1 (α9), AF-Q13002 (α10) |

### Model Comparison (9 models, ranked by Macro F1)

| Model | Macro F1 | MCC | Bal Acc | Accuracy |
|---|---|---|---|---|
| **CatBoost** | **0.483** | 0.262 | 0.506 | 0.529 |
| LightGBM | 0.461 | 0.227 | 0.484 | 0.515 |
| Random Forest | 0.461 | 0.249 | 0.484 | 0.516 |
| Logistic Regression | 0.461 | 0.253 | 0.489 | 0.512 |
| SVM (RBF) | 0.459 | 0.255 | 0.483 | 0.531 |
| XGBoost | 0.451 | 0.238 | 0.475 | 0.523 |
| MLP | 0.451 | 0.215 | 0.472 | 0.502 |
| KNN | 0.440 | 0.201 | 0.468 | 0.478 |
| Gaussian NB | 0.368 | 0.165 | 0.437 | 0.405 |


### Per-Class F1 (CatBoost, best model)

The 3-class macro F1 of 0.483 is dragged down by the 'No net effect' class which is inherently ambiguous.

| Class | F1 | Precision | Recall | Support |
|---|---|---|---|---|
| LOF | 0.52 | 0.60 | 0.50 | 218 |
| No net effect | 0.41 | 0.42 | 0.45 | 193 |
| GOF | 0.41 | 0.39 | 0.46 | 386 |


### Feature Group Ablation (CatBoost only)

Leave-one-group-out: dropping each feature group and measuring F1 change.

| Condition | F1 | MCC | Δ F1 | Verdict |
|---|---|---|---|---|
| Full model (66 features) | 0.481 | 0.276 | — | — |
| − Positional | 0.429 | 0.195 | −0.052 | 🔴 critical |
| − Physicochemical | 0.466 | 0.260 | −0.015 | 🟡 important |
| − Structural nAChR | 0.468 | 0.236 | −0.013 | 🟡 important |
| − Substitution | 0.476 | 0.273 | −0.005 | ⚪ minor |
| − Conformational | 0.479 | 0.263 | −0.002 | ⚪ negligible |
| − Structural core | 0.504 | 0.310 | +0.023 | 🟢 harmful |


### Key Takeaways (Run I)

1. **Gene identity dominates** — dropping positional features (gene one-hot) causes the largest F1 drop (-0.052). Which subunit a variant is in matters most for GOF vs LOF prediction.

2. **Core structural features are harmful** — simplified DSSP (no real mkdssp ASA values) adds noise. Installing mkdssp for real solvent accessibility would likely reverse this.

3. **nAChR-specific features add value** — TM domain, pore distance, and interface features provide modest gains beyond VEP-ENaC's feature set.

4. **3-class is hard** — the "No net effect" class has F1 ≈ 0.41 across the best model. Binary GOF/LOF collapse would likely yield F1 ~0.52-0.55.

5. **CatBoost wins on small multi-class** — ordered target encoding handles class imbalance best. Gradient boosting generally outperforms linear models on this tabular data.

6. **GPU not useful** — at 797×66, GPU kernel launch overhead dominates any compute savings.

### Comparison to VEP-ENaC (F1 ≈ 0.6)

| Factor | VEP-ENaC | VEP-nAChR2 | Impact |
|--------|----------|------------|--------|
| Task | Binary (GOF vs LOF) | 3-class (GOF/LOF/NNE) | **Major** — adding NNE class inherently reduces macro F1 |
| Data | ~400+ ENaC variants | 797 nAChR variants (3 classes) | Comparable total, but 3-way split |
| Genes | 4 ENaC genes | 16 nAChR genes | More gene diversity = harder gene-level CV |
| Structure | Simpler trimeric channel | Complex pentameric LGIC | nAChR allostery is more complex |
| Features | Full mkdssp ASA | Simplified DSSP (no real ASA) | Structural features degraded |

**Bottom line:** The 3-class task is fundamentally harder. A ~0.05 F1 gap after accounting for the extra class is expected for a more complex receptor family with incomplete structural features.

### Bugs Fixed During Run I

- Wrong PDB IDs (6UW8→9DMG, 7EKO→7KOX) — previous AI hallucinated non-nAChR structures
- AlphaFold v4→v6 URL update
- Python scoping bugs (3 instances of local imports shadowing module-level names)
- DSSP generation from CIF files (PDBe API format changed)
- XGBoost sample_weight strategy broken → switched to cost_sensitive
- BRFC too slow for nested CV → class_weight='balanced_subsample' for RF
- GPU slower than CPU on this dataset size → all models CPU-only

---
## Run II: Full Ablation — All 10 Models
**Date:** 2026-08-07  
**Config:** 797 variants, 10 models x 7 conditions (full + 6 feature group drops), 5-fold gene-level CV  
**Optuna:** 10 trials, 1 seed (reduced for ablation — feature ranking doesn't need precision)  
**GPU:** No (caused hangs on RTX 4050 + Windows)  
**Notes:** Models run directly in-process after subprocess approach hit Unicode + CPU contention timeouts

### Consensus Feature Importance (averaged across all 10 models)

Negative delta = feature helps (dropping it reduces F1). Positive delta = feature hurts (dropping it improves F1).

**Ranking (most to least important):**

| Rank | Feature Group | Avg Delta F1 | Consensus | Verdict |
|------|--------------|-------------|-----------|---------|
| 1 | **Positional** (gene OH + norm pos) | **-0.042** | 9/10 models agree | 🔴 Critical — gene identity dominates |
| 2 | Substitution (BLOSUM + Grantham) | -0.017 | 8/10 models agree | 🟡 Important |
| 3 | Physicochemical (24 AA props) | -0.016 | 8/10 models agree | 🟡 Important |
| 4 | Structural nAChR (TM, pore, interface) | -0.009 | 6/10 models agree | 🟡 Modest benefit |
| 5 | Structural core (RSA, B-factor, DSSP) | +0.004 | Mixed (5 help, 5 hurt) | ⚪ Noisy — needs mkdssp |
| 6 | Conformational (alpha7 open/closed) | +0.005 | Mixed (6 hurt, 4 help) | ⚪ Only alpha7, adds noise for others |

Δ F1 per model (negative = feature HELPS, positive = HURTS):

| Condition | CB | LGBM | RF | LR | SVM-RBF | SVM-Lin | XGB | KNN | MLP | NB | AVG | Consensus |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| drop_positional | −.075 | −.017 | −.089 | −.057 | −.054 | −.085 | −.038 | −.021 | +.002 | +.019 | **−.042** | 8 |
| drop_substitution | −.013 | −.000 | −.035 | −.021 | −.027 | −.033 | −.014 | −.002 | −.001 | −.020 | −.017 | 9 |
| drop_physicochemical | −.001 | −.012 | +.016 | −.027 | −.019 | −.033 | −.015 | −.020 | +.020 | −.065 | −.016 | 8 |
| drop_structural_nachr | −.003 | −.012 | −.050 | −.004 | −.019 | −.034 | +.005 | +.001 | +.041 | −.013 | −.009 | 7 |
| drop_structural_core | +.003 | +.014 | −.007 | +.010 | +.011 | −.019 | −.002 | +.019 | +.035 | −.028 | +.004 | 4 |
| drop_conformational | −.017 | +.036 | −.005 | +.003 | −.001 | −.003 | −.006 | +.007 | +.040 | −.009 | +.005 | 6 |


### Run II Key Takeaways

1. **Positional features are the undisputed #1** — across 9/10 models, dropping gene identity causes the largest F1 drop. Average -0.042 delta. This isn't "overfitting to gene" — it reflects real biology: different subunits have different functional roles (e.g., alpha vs beta vs delta/gamma/epsilon).

2. **Physicochemical + Substitution are tied for #2** — both contribute ~0.016-0.017 F1. These are VEP-ENaC's core features and they generalize well to nAChRs.

3. **Core structural features are noise, not signal** — 5/10 models improve when you DROP them. The simplified DSSP (no real solvent accessibility) injects noise. **Action: install mkdssp to get real ASA values for Run III.**

4. **nAChR-specific structural features help modestly** — -0.009 average delta, 6/10 models benefit. TM domain + pore distance + interface contacts capture real nAChR biology.

5. **Conformational features are dead weight** — +0.005 average (slightly harmful). Only implemented for alpha7; adds noise for the other 15 genes. Either expand to all subunits or drop entirely.

6. **MLP is most divergent** — it's the only model where dropping ANY feature group IMPROVES F1 (all deltas positive). MLP can't handle our small dataset well with random over-sampling.

7. **Gaussian NB loves physicochemical features** — -0.065 delta, by far the largest single-model impact. NB's naive conditional independence assumption is a good match for independent AA properties.

### Comparison: Run I (CatBoost only) vs Run II (all 10 models)

| Finding | Run I (CatBoost) | Run II (10-model avg) | Consistent? |
|---------|-----------------|----------------------|-------------|
| Positional is #1 | -0.052 | -0.042 | ✅ Yes |
| Physicochemical matters | -0.015 | -0.016 | ✅ Yes |
| Structural core harmful | +0.023 | +0.004 | ✅ Yes (direction same, weaker) |
| Conformational negligible | -0.002 | +0.005 | ✅ Yes (near zero both times) |

The single-model CatBoost ablation was directionally correct on every finding. Run II confirms everything with 10× the evidence.

### Planned Next Runs

- **Run III:** Install mkdssp, regenerate DSSP with real ASA, rerun ablation
- **Run IV:** Species transfer experiment (human-only vs mouse-augmented vs mixed)
- **Run V:** Binary GOF/LOF only (drop No-net-effect class)
- **Run VI:** Add ESM-2 embeddings + AlphaMissense scores

---
## Run IV: mkdssp Real ASA + 80/20 Holdout Split
**Date:** 2026-08-13
**Config:** 797 variants, 66 features, 10 models, **80/20 stratified holdout** (NOT gene-grouped), 3 seeds (42/123/456), 20 Optuna trials, inner 5-fold
**Structural fix:** Real DSSP solvent accessibility via **mkdssp 4.0.4** (canonical Kabsch & Sander), run through WSL Ubuntu — replaces the simplified CIF-fallback DSSP that had ASA=0 everywhere
**CV change:** `train_test_split(test_size=0.2, stratify=y)` — a plain random split instead of leave-one-subunit-out grouping

Two changes are bundled here, so the score jump cannot be attributed to either one alone:
1. **mkdssp real ASA** — RSA (the #1 structural feature) now has 381 unique values instead of 1. ~88-97% of residues per structure now carry genuine ASA>0 (was ~0%).
2. **80/20 holdout** — trains ~638 / tests ~159 variants, with the same subunit appearing in both train and test (gene leakage).

| Model | Macro F1 | F1 ± | MCC | Bal Acc | Accuracy |
|---|---|---|---|---|---|
| **CatBoost** | **0.6624** | 0.0075 | 0.4953 | 0.6761 | 0.6708 |
| XGBoost | 0.6605 | 0.0241 | 0.4948 | 0.6581 | 0.6854 |
| Random Forest | 0.6578 | 0.0153 | 0.4863 | 0.6632 | 0.6729 |
| LightGBM | 0.6385 | 0.0169 | 0.4576 | 0.6492 | 0.6500 |
| MLP | 0.6311 | 0.0491 | 0.4560 | 0.6334 | 0.6562 |
| Logistic Regression | 0.6122 | 0.0449 | 0.4316 | 0.6351 | 0.6188 |
| SVM (Linear) | 0.6077 | 0.0353 | 0.4258 | 0.6229 | 0.6208 |
| SVM (RBF) | 0.5935 | 0.0208 | 0.3951 | 0.5991 | 0.6146 |
| KNN | 0.5590 | 0.0219 | 0.3413 | 0.5700 | 0.5708 |
| Gaussian NB | 0.4414 | 0.0295 | 0.2498 | 0.5137 | 0.4438 |


### Run IV Key Takeaways

**⚠️ These scores are an upper bound (ceiling), not a generalization estimate.**

The +0.18 F1 jump over Run I (CatBoost 0.483 → 0.662) is driven almost entirely by the **80/20 holdout split**, not by the mkdssp feature fix:

1. **Gene leakage inflates every score.** With a random split, the same subunit (gene) appears in both train and test. The gene one-hot — the #1 most important feature per both Run I and Run II ablations — can be *memorized* per subunit. The model learns "CHRNA7 variants tend to be GOF" and re-applies that to CHRNA7 test variants it already saw in training.

2. **This is why the ranking is so stable** — CatBoost 0.662 vs Gaussian NB 0.441 still spans 0.22 F1, and the model ordering is identical to Run I. The split change lifts everyone up uniformly; it doesn't change *which* model wins.

3. **The mkdssp ASA fix is NOT isolated here.** To measure the structural-feature effect alone, we'd need to rerun the **gene-grouped CV** with real ASA and compare against Run II. (Run III already tested this with freesasa ASA and found **no improvement** — structural features are not the bottleneck; gene identity is.)

4. **For the thesis, use Run I/II numbers as the honest generalization estimate.** The holdout numbers are a sanity check ("can the model fit its own data?") but gene-grouped CV is the only defensible metric for a novel-subunit generalization claim.

### Run IV vs Run I (CatBoost)

| Metric | Run I (gene-grouped) | Run IV (80/20 holdout) | Δ |
|--------|---------------------|------------------------|---|
| Macro F1 | 0.483 | 0.662 | **+0.179** |
| MCC | 0.262 | 0.495 | +0.233 |
| Bal Acc | 0.506 | 0.676 | +0.170 |
| Accuracy | 0.529 | 0.671 | +0.142 |

### Next steps (revised)

- **Run V (priority):** Isolate the mkdssp effect — rerun gene-grouped CV with real ASA, compare to Run II to see if structural features now help.
- **Run VI:** Binary GOF/LOF only (drop No-net-effect class).
- **Run VII:** Species transfer experiment (human-only vs mouse-augmented vs mixed).
- **Run VIII:** Add ESM-2 embeddings + AlphaMissense scores.

---
## Run V: mkdssp Real ASA + Gene-Grouped CV (isolates the mkdssp effect)
**Date:** 2026-08-15
**Config:** 797 variants, 66 features, 10 models, **subunit CV** (leave-one-subunit-out `StratifiedGroupKFold` — no gene leakage), 3 seeds (42/123/456), 20 Optuna trials
**Change vs Run I/II:** real mkdssp 4.0.4 ASA (was simplified CIF-fallback with ASA=0 everywhere)
**Change vs Run IV:** gene-grouped split restored (was 80/20 holdout)

This is the apples-to-apples test for the mkdssp fix: same gene-grouped CV as Run I/II, but with real solvent accessibility. Any delta vs Run I is *purely* the structural-feature fix — no split confound. Total wall-clock 155 min.

| Model | Macro F1 | F1 ± | MCC | Bal Acc | Accuracy |
|---|---|---|---|---|---|
| **CatBoost** | **0.4891** | 0.0206 | 0.2743 | 0.5189 | 0.5255 |
| Random Forest | 0.4701 | 0.0235 | 0.2598 | 0.4932 | 0.5194 |
| LightGBM | 0.4663 | 0.0192 | 0.2514 | 0.4939 | 0.5168 |
| Logistic Regression | 0.4611 | 0.0245 | 0.2460 | 0.4928 | 0.4925 |
| SVM (Linear) | 0.4464 | 0.0158 | 0.2195 | 0.4743 | 0.4834 |
| XGBoost | 0.4444 | 0.0335 | 0.2204 | 0.4686 | 0.5028 |
| SVM (RBF) | 0.4291 | 0.0191 | 0.1985 | 0.4598 | 0.4782 |
| KNN | 0.4259 | 0.0143 | 0.1675 | 0.4628 | 0.4598 |
| MLP | 0.4231 | 0.0213 | 0.1799 | 0.4470 | 0.4688 |
| Gaussian NB | 0.3867 | 0.0444 | 0.1837 | 0.4456 | 0.4258 |


### Run V Key Takeaways

**The mkdssp real-ASA fix does NOT move the needle.** This was the clean isolate test, and the answer is now definitive.

| Model | Run I (simplified DSSP) | Run V (mkdssp ASA) | Δ F1 |
|-------|------------------------|--------------------|------|
| CatBoost | 0.483 | 0.489 | +0.006 |
| Random Forest | 0.461 | 0.470 | +0.009 |
| LightGBM | 0.461 | 0.466 | +0.005 |
| Logistic Regression | 0.461 | 0.461 | 0.000 |
| SVM (RBF) | 0.459 | 0.429 | -0.030 |
| XGBoost | 0.451 | 0.444 | -0.007 |
| MLP | 0.451 | 0.423 | -0.028 |
| KNN | 0.440 | 0.426 | -0.014 |
| Gaussian NB | 0.368 | 0.387 | +0.019 |
| **Mean** | | | **-0.004** |

1. **Mean Δ ≈ -0.004 F1** — statistically indistinguishable from zero. Half the models improve a hair, half get a hair worse. This is noise, not signal.

2. **Confirms the Run II ablation.** Run II already flagged `structural_core` as neutral (+0.004 delta, "mixed: 5 help, 5 hurt"). Giving RSA its correct values doesn't change that — it stays neutral. The simplified DSSP was *not* the thing holding the model back.

3. **The bottleneck is elsewhere.** The inner-CV "Best inner F1" hits ~0.65 while held-out-subunit test F1 sits at ~0.49. That ~0.16 gap is the generalization ceiling from gene-grouped CV, and it's driven by gene identity (positional features, -0.042 in Run II), not by missing structural detail.

4. **Verdict for the thesis:** keep the mkdssp real ASA (it's scientifically defensible — RSA values are now physically meaningful), but do *not* expect it to improve GOF/LOF prediction. The structural-feature question is answered: it's a neutral feature group for this task. To move F1, the levers are elsewhere — binary GOF/LOF collapse (drop the ambiguous "No net effect" class), more data (mouse augmentation), or embeddings (ESM-2/AlphaMissense).

5. **The honest generalization number stays ~0.49.** Run IV's 0.66 was the leakage ceiling; Run V's 0.49 (CatBoost) is the real number to report.

### Run IV vs Run V — the split effect, quantified

| Model | Run IV (80/20 holdout) | Run V (gene-grouped) | Δ (leakage) |
|-------|------------------------|----------------------|-------------|
| CatBoost | 0.662 | 0.489 | +0.173 |
| Gaussian NB | 0.441 | 0.387 | +0.055 |
| XGBoost | 0.661 | 0.444 | +0.216 |

The 80/20 holdout inflates F1 by **+0.17 to +0.22** across models — this is the pure gene-leakage premium, now measured directly.

---
## Run VI: Binary GOF/LOF (drop No-net-effect class)
**Date:** 2026-08-16
**Config:** 594 variants (452 human / 87 rat / 55 mouse), 66 features, 10 models, **subunit CV** (gene-grouped, no leakage), 3 seeds (42/123/456), 20 Optuna trials
**Change vs Run V:** dropped the "No net effect" class → binary GOF (208) vs LOF (386). Same CV, same features, same seeds.
**Also fixed:** loader.py:112 NaN-species bug + dropped 9 rows with positions outside the human reference length (data-entry errors e.g. CHRNA1 p=1314, CHRNE p=-8).

This tests the Run II/V hypothesis that the ambiguous "No net effect" class was dragging down macro F1. Removing it isolates the GOF-vs-LOF signal — the scientifically interesting question, and the task MissION / VEP-ENaC actually solve.

| Model | Macro F1 | F1 ± | MCC | Bal Acc | Accuracy |
|---|---|---|---|---|---|
| **Random Forest** | **0.6445** | 0.0378 | 0.3268 | 0.6640 | 0.6928 |
| KNN | 0.6335 | 0.0065 | 0.2997 | 0.6547 | 0.6720 |
| LightGBM | 0.6289 | 0.0150 | 0.2886 | 0.6425 | 0.6675 |
| CatBoost | 0.6289 | 0.0354 | 0.2949 | 0.6486 | 0.6706 |
| XGBoost | 0.6283 | 0.0415 | 0.3111 | 0.6477 | 0.6850 |
| SVM (RBF) | 0.5942 | 0.0502 | 0.2375 | 0.6149 | 0.6435 |
| MLP | 0.5907 | 0.0064 | 0.2188 | 0.6099 | 0.6339 |
| Logistic Regression | 0.5771 | 0.0281 | 0.2014 | 0.5997 | 0.6173 |
| SVM (Linear) | 0.5660 | 0.0727 | 0.1911 | 0.5868 | 0.6067 |
| Gaussian NB | 0.5006 | 0.0542 | 0.1461 | 0.5758 | 0.5198 |


### Run VI vs Run V — every model improves

Dropping the "No net effect" class lifts macro F1 for all 10 models by **+0.11 to +0.21** (mean +0.155).

⚠️ **Caveat:** macro F1 over 2 classes is not strictly comparable to macro F1 over 3 classes — the 3-class metric is penalized by an extra (harder) class, so part of the jump is arithmetic rather than signal. The clean read: **the NNE class was the single hardest class to predict and was suppressing the headline number.** Removing it surfaces the GOF/LOF skill that was always there underneath.

**Binary baselines:** always-predict-LOF (majority) = 0.394 macro F1; chance ≈ 0.50. Random Forest's 0.644 (MCC 0.327, Bal Acc 0.664) is real but modest signal — roughly in line with VEP-ENaC's binary ~0.6 reference.

### Key Takeaways

1. **Binary GOF/LOF is the right framing** — it's what MissION and VEP-ENaC actually solve, and it lifts every model by 0.11–0.21 macro F1. For the thesis, report binary as the primary task and keep 3-class as the harder secondary.

2. **Random Forest takes the crown** (0.644), with KNN/LightGBM/CatBoost/XGBoost clustered at ~0.63. Gaussian NB (0.50) is still worst — its naive independence assumption remains a poor fit.

3. **Model ranking is stable vs Run V** — RF, LightGBM, CatBoost, XGBoost stay near the top; Gaussian NB stays last. The class collapse lifts everyone uniformly without reshuffling the winner.

4. **⚠️ 142 rodent rows (87 rat + 55 mouse) are still contaminated** — they feed rodent residue numbers into human references/PDBs. That noise is still inside Run VI. After the cross-species remap (see Section: cross-species fix), the number may shift again (likely up).

5. **The remaining levers are data + chemistry, not structure.** mkdssp is confirmed neutral (Run V) and NNE is removed (Run VI). What's left: fix the broken `structural_nachr` features, add ThermoMPNN ΔΔG / ESM-2 embeddings, and correctly remap the rodent data.

| Model | Run V (3-class) | Run VI (binary) | Δ F1 |
|---|---|---|---|
| Random Forest | 0.4701 | 0.6445 | +0.174 |
| KNN | 0.4259 | 0.6335 | +0.208 |
| LightGBM | 0.4663 | 0.6289 | +0.163 |
| CatBoost | 0.4891 | 0.6289 | +0.140 |
| XGBoost | 0.4444 | 0.6283 | +0.184 |
| SVM (RBF) | 0.4291 | 0.5942 | +0.165 |
| MLP | 0.4231 | 0.5907 | +0.168 |
| Logistic Regression | 0.4611 | 0.5771 | +0.116 |
| SVM (Linear) | 0.4464 | 0.5660 | +0.120 |
| Gaussian NB | 0.3867 | 0.5006 | +0.114 |


---
## Run VII: Cross-Species + Numbering Normalization (final_mapped.xlsx)
**Date:** 2026-08-19
**Config:** 717 variants (was 785), 66 features, 10 models, subunit CV (gene-grouped, no leakage), 3 seeds (42/123/456), 30 Optuna trials
**Change vs Run V:** the "AA position" column re-anchored to the human precursor RefSeq numbering, and mouse/rat rows remapped onto the human reference.

### What was broken

`final.xlsx` stored positions in a **mixed numbering frame** — some genes in precursor (full-length) numbering, others in mature-protein numbering (signal peptide stripped, offset +20–33), plus a few data-entry typos. This was a *pre-existing* bug affecting roughly half the **human** data too, not just the rodent orthologs. Every position-keyed feature — normalized position, TM-helix, DSSP/RSA, B-factor, pore distance — was silently ~20–33 residues off for those rows.

### The fix

`scripts/normalize_positions.py` anchors each row back onto the reference using its **wildtype amino acid** (the one reliable ground truth papers report). Resolution order: precursor → mature (+signal peptide) → typo (±5) → isoform (+15..40) → drop.

| Outcome | Rows |
|---------|------|
| Resolved to human precursor numbering | 755 / 841 (89.8%) |
| Dropped (position blanked) | 86 / 841 (10.2%) |

Status breakdown: mature 352, precursor 161, mature+ortholog 115, precursor+ortholog 62, typo 19, isoform+ortholog 17, typo+ortholog 17, isoform 12. Dropped: 73 ambiguous + 11 no_match + 1 no_reference + 1 missing.

| Dataset | Variants | Human / Rat / Mouse |
|---------|----------|---------------------|
| Baseline (`final.xlsx`) | 785 | 533 / 174 / 78 |
| Mapped (`final_mapped.xlsx`) | 717 | 507 / 138 / 72 |

Net −68 variants (8.7%), but every surviving row now carries a correct precursor position.

*Mean macro F1 — Run V: 0.444 | Run VII: 0.448 | Δ +0.004*

| Model | Run V (unmapped) | Run VII (mapped) | Δ F1 |
|---|---|---|---|
| Random Forest | 0.4701 | 0.4834 | +0.013 |
| LightGBM | 0.4663 | 0.4628 | −0.004 |
| XGBoost | 0.4444 | 0.4599 | +0.016 |
| MLP | 0.4231 | 0.4578 | +0.035 |
| Logistic Regression | 0.4611 | 0.4534 | −0.008 |
| CatBoost | 0.4891 | 0.4531 | −0.036 |
| SVM (Linear) | 0.4464 | 0.4472 | +0.001 |
| KNN | 0.4259 | 0.4472 | +0.021 |
| SVM (RBF) | 0.4291 | 0.4370 | +0.008 |
| Gaussian NB | 0.3867 | 0.3768 | −0.010 |


### Run VII Key Takeaways

**The normalization is F1-neutral (mean Δ = +0.004).** The hypothesis that fixing the residue positions would lift the score did not hold — 7 of 10 models moved within ±0.02 (noise), and the mean is essentially unchanged. The honest headline: the mapping does **not** improve generalization macro F1.

1. **CatBoost collapsed (−0.036).** The Run V champion (0.489) fell to 0.453. Its prior edge was partly an artifact of the noisy positions (and a favorable fold split) rather than robust signal. With corrected positions *and* 68 fewer training rows, it falls back into the pack.

2. **Random Forest becomes the new best (0.483).** RF (+0.013), XGBoost (+0.016), MLP (+0.035), and KNN (+0.021) all improved. The leaderboard is now flatter and RF — already the binary-mode winner in Run VI — is also the 3-class winner.

3. **This is exactly what the Run II/V ablations predicted.** Position-dependent features (normalized position, structural) are near-neutral (structural_core +0.004, structural_nachr −0.009). The model's signal is dominated by **gene one-hot + physicochemical properties**, neither of which depends on the residue position being correct. Fixing the input to already-neutral features cannot move the output.

4. **Keep the mapping anyway.** The data is now scientifically correct: a variant reported as "mature residue 50" is stored as its true precursor position, and every structural feature now reads the right residue. For the thesis this is non-negotiable correctness — even though it doesn't change the headline F1.

5. **The real ceiling is still gene-level generalization.** The inner-CV "best inner F1" (~0.65) vs held-out-subunit test F1 (~0.45–0.48) gap is unchanged. The levers that actually matter remain: more data (cross-species augmentation), embeddings (ESM-2/AlphaMissense/ThermoMPNN), or reframing the task — not residue numbering.

### Run VII vs Run V — the honest read

| Metric | Run V (unmapped) | Run VII (mapped) | Δ |
|--------|-----------------|------------------|---|
| Mean macro F1 (10 models) | 0.444 | 0.448 | +0.004 |
| Best model F1 | 0.489 (CatBoost) | 0.483 (Random Forest) | −0.006 |
| Data integrity | mixed precursor/mature frame | all precursor, WT-AA verified | ✅ fixed |

### Run VII — Binary GOF/LOF: same normalization, same (null) result
**Config:** 553 mapped variants (GOF 190 / LOF 363), binary task, subunit CV, 3 seeds (42/123/456), 30 Optuna trials — identical settings to Run VI but on `final_mapped.xlsx`.
**Change vs Run VI:** the same numbering normalization as the 3-class run above (plus the loader's NaN-species + out-of-range fixes already present in Run VI).

*Mean macro F1 — Run VI: 0.599 | Run VII (binary): 0.600 | Δ +0.001*

| Model | Run VI (unmapped) | Run VII (mapped) | Δ F1 |
|---|---|---|---|
| XGBoost | 0.6283 | 0.6411 | +0.013 |
| CatBoost | 0.6289 | 0.6303 | +0.001 |
| Random Forest | 0.6445 | 0.6291 | −0.015 |
| Logistic Regression | 0.5771 | 0.6070 | +0.030 |
| LightGBM | 0.6289 | 0.6020 | −0.027 |
| KNN | 0.6335 | 0.6012 | −0.032 |
| SVM (Linear) | 0.5660 | 0.5924 | +0.026 |
| MLP | 0.5907 | 0.5867 | −0.004 |
| SVM (RBF) | 0.5942 | 0.5611 | −0.033 |
| Gaussian NB | 0.5006 | 0.5497 | +0.049 |


### Run VII (binary) Key Takeaways

**Binary is also a wash — mean Δ = +0.001.** The numbering normalization is F1-neutral in *both* framings. This closes the loop on Run VI's open question: it predicted "the number may shift again (likely up)" after the rodent remap; in fact it shifted by ~nothing.

1. **XGBoost takes the binary crown (0.641),** overtaking Random Forest (0.644 → 0.629). The two are statistically indistinguishable (Δ +0.013 vs −0.015, both within the ±0.03–0.05 fold noise).

2. **The top binary models dipped, the mid-tier rose.** RF −0.015, LightGBM −0.027, KNN −0.032; but LR +0.030, SVM-Linear +0.026, Gaussian NB +0.049. Net: flat. Removing the contaminated rodent positions didn't remove any *usable* signal — the model was already discounting those rows.

3. **Gaussian NB's +0.049 jump is the largest single move** but irrelevant — it remains the worst model (0.550). Its naive independence assumption is the bottleneck, not the positions.

4. **Conclusion across both framings:** residue-numbering correctness is a **data-integrity** win, not a **predictive** win. The F1 ceiling is set by gene-level generalization (the ~0.65 inner-CV → ~0.48–0.64 test gap) and by data volume — not by where the residues are numbered. To move the number, the levers are more data (mouse/rat augmentation at scale), embeddings (ESM-2 / AlphaMissense / ThermoMPNN ΔΔG), or a different feature set.

---
## Run VIII: + Position Conservation & ESM-2 Zero-Shot Features (66 → 71 features)
**Date:** 2026-08-21
**Config:** 717 variants, **71 features**, 10 models, subunit CV (gene-grouped, no leakage), 3 seeds (42/123/456), 30 Optuna trials
**Change vs Run VII:** two new feature groups added — (1) **position-specific evolutionary conservation** from the mouse/rat ortholog alignments, and (2) **ESM-2 protein language model** zero-shot variant-effect scores.

### The two new features (and why they target the real bottleneck)

Run VII concluded the F1 ceiling is set by *gene-level generalization*, and named "embeddings + more data" as the levers that would actually move the number. These two features attack that ceiling directly — without touching the data pool (still 717 variants):

1. **Conservation (3 features: `conservation_wt`, `conservation_mt`, `conservation_delta`).** BLOSUM62/Grantham measure *how big* the amino-acid change is — position-independent. Conservation instead measures *how constrained this exact residue* is: `conservation_wt` = fraction of aligned ortholog columns (mouse/rat/human) carrying the wildtype AA at that position, `conservation_mt` = the same for the mutant, `conservation_delta` = the drop. Built entirely from the precomputed ortholog mappings (`mapping/{mouse,rat}_to_human.csv`) — **no new downloads**. Mean `conservation_wt` = 0.936 (std 0.181): nAChR residues are, on average, strongly conserved.

2. **ESM-2 (2 features: `esm2_wt_logprob`, `esm2_effect`).** The masked-marginal zero-shot score (Meier et al. 2021): mask the variant position, run ESM-2 (`esm2_t30_150M_UR50D`), and read `log P(mutant | context) − log P(wildtype | context)`. `esm2_wt_logprob` is an ESM-based conservation proxy (how "surprising" the wildtype residue is); `esm2_effect` is the zero-shot mutation effect (negative = disfavoured). Range [−12.4, +6.4], mean −3.1. CPU-friendly 150M checkpoint, loaded lazily and cached.


*Mean macro F1 — Run VII: 0.448 | Run VIII: 0.471 | Δ +0.023*

| Model | Run VII (66 feats) | Run VIII (71 feats) | Δ F1 |
|---|---|---|---|
| CatBoost | 0.4531 | 0.5081 | +0.055 |
| Random Forest | 0.4834 | 0.5012 | +0.018 |
| Logistic Regression | 0.4534 | 0.4834 | +0.030 |
| LightGBM | 0.4628 | 0.4750 | +0.012 |
| SVM (Linear) | 0.4472 | 0.4745 | +0.027 |
| XGBoost | 0.4599 | 0.4695 | +0.010 |
| MLP | 0.4578 | 0.4669 | +0.009 |
| KNN | 0.4472 | 0.4651 | +0.018 |
| Gaussian NB | 0.3768 | 0.4586 | +0.082 |
| SVM (RBF) | 0.4370 | 0.4099 | −0.027 |


### Run VIII (3-class) Key Takeaways

**The features work — mean Δ = +0.023, 9/10 models improve, new best CatBoost 0.508.** Unlike the numbering normalization (Run VII, F1-neutral), adding conservation + ESM-2 *did* move the number. This is the first feature-driven improvement of the whole project — every prior run moved F1 through data fixes or task reframing, not signal.

1. **New champion: CatBoost 0.508 (+0.055).** It jumps from 0.453 (7th) to 0.508 (1st), overtaking Random Forest (0.483 → 0.501). The single biggest move in the table — conservation + ESM-2 carry signal that ordered boosting can exploit.

2. **The weak models surged, the field compressed.** Gaussian NB +0.082 (its best-ever, though still last), LR +0.030, SVM-Linear +0.027, CatBoost +0.055. The additions inject *positional/constraint* signal that linear and naive models previously couldn't see through the gene one-hot + physicochemical features.

3. **One regression: SVM-RBF (−0.027),** and its fold std roughly doubled to 0.045 (highest in the table). The ESM-2 log-probs are heavy-tailed (−12…+6, clustered near −3). RBF does get `RobustScaler` (median/IQR) via `needs_scaling=True`, but that doesn't fully tame a heavy tail — the RBF kernel's squared distance can still be dominated by those tail values. Likely a scaling/tail interaction, not a signal problem. Worth checking by winsorizing/clipping or standardizing the ESM columns, or dropping them for the RBF model alone.

4. **The ceiling is budging, but only just.** 0.508 is still far below the inner-CV ~0.65 and not yet usable. Conservation + ESM-2 are the *cheap* version of the "embeddings + more data" lever Run VII named — and they now confirm that lever is the right direction. Next: full embeddings (ESM-2 650M, AlphaMissense, ThermoMPNN ΔΔG) and scaling the mouse/rat augmentation.

### Run VIII vs Run VII — 3-class

| Metric | Run VII (66 feats) | Run VIII (71 feats) | Δ |
|--------|-------------------|---------------------|---|
| Mean macro F1 (10 models) | 0.448 | 0.471 | +0.023 |
| Best model F1 | 0.483 (Random Forest) | 0.508 (CatBoost) | +0.025 |
| Models improved | — | 9 / 10 | ✅ |

*(Binary subunit, 3-class holdout, and binary holdout results follow below.)*


### Run VIII — Binary GOF/LOF: the same features do *not* help the direction task
**Config:** 553 mapped variants (GOF 190 / LOF 363), binary task, **71 features**, subunit CV, 3 seeds (42/123/456), 30 Optuna trials — identical settings to Run VII binary, plus conservation + ESM-2.
**Change vs Run VII binary:** the two new feature groups (conservation + ESM-2), same as the 3-class run above.


*Mean macro F1 — Run VII bin: 0.600 | Run VIII bin: 0.604 | Δ +0.004*

| Model | Run VII bin (66) | Run VIII bin (71) | Δ F1 |
|---|---|---|---|
| Random Forest | 0.6291 | 0.6469 | +0.018 |
| CatBoost | 0.6303 | 0.6366 | +0.006 |
| XGBoost | 0.6411 | 0.6245 | −0.017 |
| LightGBM | 0.6020 | 0.6169 | +0.015 |
| Logistic Regression | 0.6070 | 0.5978 | −0.009 |
| MLP | 0.5867 | 0.5955 | +0.009 |
| KNN | 0.6012 | 0.5858 | −0.015 |
| Gaussian NB | 0.5497 | 0.5853 | +0.036 |
| SVM (Linear) | 0.5924 | 0.5793 | −0.013 |
| SVM (RBF) | 0.5611 | 0.5675 | +0.006 |


### Run VIII (binary) Key Takeaways

**Binary is flat — mean Δ = +0.004, only 6/10 models improve.** The same conservation + ESM-2 features that lifted 3-class by +0.023 do essentially *nothing* for GOF/LOF direction. This asymmetry is itself the finding.

1. **The features separate "effect vs no-effect", not "GOF vs LOF".** Ortholog conservation and ESM-2's masked-marginal score both answer *"how much does this mutation perturb the protein?"* — which maps cleanly onto the 3-class task's No-net-effect vs damaging axis. But a GOF mutation and a LOF mutation are **both** "damaging/constrained" from that viewpoint, so the features are blind to the *direction*. The 3-class gain was almost entirely the model getting better at spotting No-net-effect; the GOF↔LOF split it already could make is unchanged.

2. **Best binary model shifts within noise:** Random Forest 0.647 nudges past XGBoost's 0.641 baseline — the two have swapped places across every run and are statistically indistinguishable (±0.03–0.05 fold noise).

3. **This is the clearest evidence yet that direction-of-effect is a distinct, harder problem.** Generic pathogenicity/constraint features (ESM-2, conservation — and by extension AlphaMissense, PolyPhen, CADD) can triage *whether* a variant matters but not *which way* it pushes the receptor. That is precisely the gap the project exists to fill — and why a nAChR-specific GOF/LOF model is not just a re-labelled pathogenicity classifier.

### Run VIII vs Run VII — binary

| Metric | Run VII bin (66) | Run VIII bin (71) | Δ |
|--------|-----------------|-------------------|-----|
| Mean macro F1 (10 models) | 0.600 | 0.604 | +0.004 |
| Best model F1 | 0.641 (XGBoost) | 0.647 (Random Forest) | +0.006 |
| Models improved | — | 6 / 10 | ~noise |

*(Holdout results follow below.)*


### Run VIII — 3-class Holdout: confounded comparison, but the direction is telling
**Config:** 717 mapped variants, **71 features**, 3-class, **holdout (random train/test split — gene-leaky)**, 3 seeds (42/123/456), 30 Optuna trials.
**⚠️ Caveat:** the holdout baseline (`comparison_holdout.csv`, 2026-08-14) was generated **before** the numbering normalization (unmapped, 785 variants), and there is no mapped 66-feature holdout run to compare against. This table therefore conflates **two** changes — the normalization (−68 rows) *and* the new features. The subunit CV above is the controlled comparison; read this one directionally, not as a clean ablation.


*Mean macro F1 — base holdout: 0.606 | holdout + features: 0.599 | Δ −0.008*

| Model | Base holdout (66, unmapped) | Holdout + features (71, mapped) | Δ F1 |
|---|---|---|---|
| Random Forest | 0.6578 | 0.6411 | −0.017 |
| CatBoost | 0.6624 | 0.6409 | −0.022 |
| Logistic Regression | 0.6122 | 0.6159 | +0.004 |
| SVM (Linear) | 0.6077 | 0.6152 | +0.008 |
| SVM (RBF) | 0.5935 | 0.6079 | +0.014 |
| LightGBM | 0.6385 | 0.6059 | −0.033 |
| XGBoost | 0.6605 | 0.6042 | −0.056 |
| MLP | 0.6311 | 0.5949 | −0.036 |
| KNN | 0.5590 | 0.5518 | −0.007 |
| Gaussian NB | 0.4414 | 0.5093 | +0.068 |


### Run VIII (holdout) Key Takeaways

**Holdout mean −0.008, and the pattern is a mirror-image of subunit.** Only 4/10 models improve, and the *trees* — not the weak models — are the ones that dropped. Read with the confound caveat above; the interpretation is still useful:

1. **Trees dropped, weak models rose.** XGBoost −0.056, MLP −0.036, LightGBM −0.033, CatBoost −0.022, RF −0.017; but Gaussian NB +0.068, SVM-RBF +0.014, SVM-Linear +0.008. The *"features help the weak models"* signature from subunit CV is still visible — it's just swamped on the tree side by the other change (normalization).

2. **Why the trees dropped: leakage removal, not signal loss.** In a gene-leaky random split, the pre-normalization tree models were exploiting the *noisy residue positions* as a free gene-identity proxy (a variant's mis-numbered position partially encodes which gene it came from). The normalization stripped that leak, so the leaky holdout score falls back toward the honest number. This is the same leakage Run VII documented, just seen from the other side.

3. **This is exactly why the project uses gene-grouped CV.** The holdout (~0.60–0.66) is the optimistic, VEP-ENaC-style number; the subunit CV (~0.47–0.51) is the honest one. The features' real effect (+0.023 on subunit) is invisible in this confounded holdout, which is *fine* — the holdout was never the metric that matters.


### Run VIII — Binary Holdout: leaky ceiling, no baseline to compare
**Config:** 553 mapped variants (GOF 190 / LOF 363), binary task, **71 features**, **holdout (random split, gene-leaky)**, 3 seeds (42/123/456), 30 Optuna trials.
**Note:** there is **no** pre-normalization binary holdout baseline (this mode was never run on the old data), so this is absolute numbers only — the leaky/optimistic ceiling for the binary task.


*Mean binary holdout F1: 0.705 (leaky ceiling)*

| Model | Binary holdout F1 (leaky) |
|---|---|
| KNN | 0.760 |
| XGBoost | 0.733 |
| LightGBM | 0.729 |
| SVM (RBF) | 0.728 |
| CatBoost | 0.708 |
| Random Forest | 0.704 |
| MLP | 0.704 |
| SVM (Linear) | 0.696 |
| Logistic Regression | 0.665 |
| Gaussian NB | 0.618 |


### Run VIII (binary holdout) Key Takeaways

**Leaky binary ceiling ≈ 0.70 (best KNN 0.760), vs honest binary subunit ≈ 0.60.** The ~0.10 gap is the same gene-leakage Run VII documented: a random split lets the model score "same gene → same effect" on its own test rows.

1. **KNN winning (0.760) is a leaky-split artifact**, not a real champion — nearest neighbours literally find same-gene variants in the train fold. XGBoost (0.733), LightGBM (0.729), SVM-RBF (0.728) are the meaningful leaders here, and they cluster tightly.

2. **No before/after is possible** (this mode was never run pre-normalization), so this section is a ceiling reference, not a comparison. It is the number to quote if anyone asks "what's the optimistic (VEP-ENaC-style) score?" — and the reason the project's headline metric stays on the honest subunit CV.


---
## Run VIII — Overall: a real, partial win for the honest metric

| Framing | CV | Clean? | Mean Δ F1 | Best (Run VIII) |
|---|---|---|---|---|
| 3-class | subunit (honest) | ✅ | **+0.023** | CatBoost 0.508 |
| Binary GOF/LOF | subunit (honest) | ✅ | +0.004 | Random Forest 0.647 |
| 3-class | holdout (leaky) | ⚠️ confounded | −0.008 | Random Forest 0.641 |
| Binary GOF/LOF | holdout (leaky) | — (no baseline) | — | KNN 0.760 |

**The single finding.** Conservation + ESM-2 lift the honest 3-class macro-F1 by **+0.023 (9/10 models)** — the first feature-driven improvement of the entire project — but leave binary GOF/LOF unchanged (+0.004). The reason is coherent: both features answer *"how much does this mutation perturb the protein?"*, which cleanly separates **effect vs no-effect** (the 3-class No-net-effect axis) but is blind to **GOF vs LOF direction** (both read as "damaging"). That asymmetry is direct experimental evidence that direction-of-effect is orthogonal to pathogenicity — and that generic scores (ESM-2, conservation, and by extension AlphaMissense / PolyPhen / CADD) cannot substitute for a nAChR-specific GOF/LOF model.

**Housekeeping.** (1) SVM-RBF regressed in 3-class (−0.027, std 0.045) — a likely heavy-tail/scale interaction with the ESM-2 log-probs; worth winsorizing or standardizing the ESM columns. (2) The 3-class holdout comparison is confounded (its baseline predates the normalization) — the subunit CV is the controlled number.

**Next levers (now better-motivated).** Full embeddings (ESM-2 650M, AlphaMissense, ThermoMPNN ΔΔG) and cross-species augmentation at scale — both target the same *effect-vs-no-effect* axis these two cheap features already moved. Crucially, neither will close the *direction* gap: that needs nAChR-specific structure/mechanism signal (open-vs-closed conformational state, ligand/interface context), which is the project's actual contribution.


---
## Run IX: ESM-2 150M → 650M (null result — bigger model buys nothing)
**Date:** 2026-08-21
**Config:** 717 variants, 71 features, 10 models, subunit CV (gene-grouped, no leakage), 3 seeds (42/123/456), 30 Optuna trials — identical to Run VIII, but the two ESM-2 zero-shot features are recomputed from `esm2_t33_650M_UR50D` (2.6 GB) instead of `esm2_t30_150M_UR50D`.
**Change vs Run VIII:** only the ESM model size changed (150M → 650M). Same masked-marginal method, same 2 features, same 71 total. A clean A/B.


*Mean macro F1 — 150M: 0.471 | 650M: 0.465 | Δ −0.006*

| Model | 150M (Run VIII) | 650M (Run IX) | Δ F1 |
|---|---|---|---|
| Random Forest | 0.5012 | 0.5102 | +0.009 |
| CatBoost | 0.5081 | 0.4830 | −0.025 |
| LightGBM | 0.4750 | 0.4744 | −0.001 |
| KNN | 0.4651 | 0.4664 | +0.001 |
| XGBoost | 0.4695 | 0.4650 | −0.005 |
| SVM (Linear) | 0.4745 | 0.4647 | −0.010 |
| Logistic Regression | 0.4834 | 0.4561 | −0.027 |
| Gaussian NB | 0.4586 | 0.4498 | −0.009 |
| MLP | 0.4669 | 0.4457 | −0.021 |
| SVM (RBF) | 0.4099 | 0.4339 | +0.024 |


### Run IX (3-class) Key Takeaways

**650M ≈ 150M — mean Δ = −0.006, 3/10 models improve, best model unchanged (0.510 RF vs 0.508 CatBoost).** Scaling the ESM-2 checkpoint 4.3× does not move the honest metric. This is a clean A/B: only the ESM model size changed.

1. **The per-model swings are pure noise.** CatBoost −0.025, LR −0.027, MLP −0.021, but SVM-RBF +0.024, RF +0.009. Every one sits within the ±0.02–0.03 fold/seed std we see run-to-run; the −0.006 mean is indistinguishable from zero.

2. **Zero-shot score *quality* was never the bottleneck.** 150M and 650M rank "is this mutation damaging?" near-identically — the extra parameters refine fine-grained likelihoods that don't change the coarse GOF/LOF/no-net signal. This closes the "bigger LM = better features" hypothesis: the lever is the **axis** (effect-vs-no-effect vs direction) and **data volume**, not model capacity.

3. **Keep 150M.** It's ~4× faster on extraction (~17 min vs ~1.3 h) and equally good; the 650M end-to-end was ~2× slower (~5 h vs ~2 h for 3-class subunit). Further embedding work should chase *different* signals (structure-based, ΔΔG) — not bigger LMs.

### Run IX vs Run VIII — 3-class

| Metric | 150M (Run VIII) | 650M (Run IX) | Δ |
|--------|----------------|---------------|---|
| Mean macro F1 (10 models) | 0.471 | 0.465 | −0.006 |
| Best model F1 | 0.508 (CatBoost) | 0.510 (Random Forest) | +0.002 |
| Models improved | — | 3 / 10 | noise |


### Run IX — Binary GOF/LOF: 650M is flat too (Δ −0.0002)
**Config:** 553 mapped variants (GOF 190 / LOF 363), binary task, **71 features** with the two ESM-2 zero-shot features recomputed from `esm2_t33_650M_UR50D`, subunit CV (gene-grouped, no leakage), 3 seeds (42/123/456), 30 Optuna trials — identical to Run VIII binary.
**Change vs Run VIII binary:** only the ESM-2 checkpoint changed (150M → 650M). A clean A/B on the direction task.


*Mean macro F1 — 150M: 0.604 | 650M: 0.603 | Δ −0.000*

| Model | 150M (Run VIII) | 650M (Run IX) | Δ F1 |
|---|---|---|---|
| Random Forest | 0.6469 | 0.6387 | −0.008 |
| XGBoost | 0.6245 | 0.6243 | −0.000 |
| SVM (Linear) | 0.5793 | 0.6099 | +0.031 |
| MLP | 0.5955 | 0.6043 | +0.009 |
| LightGBM | 0.6169 | 0.6049 | −0.012 |
| CatBoost | 0.6366 | 0.6015 | −0.035 |
| Logistic Regression | 0.5978 | 0.5997 | +0.002 |
| KNN | 0.5858 | 0.5967 | +0.011 |
| Gaussian NB | 0.5853 | 0.5861 | +0.001 |
| SVM (RBF) | 0.5675 | 0.5682 | +0.001 |


### Run IX (binary) Key Takeaways

**Binary is flat too — mean Δ = −0.0002, 6/10 models improve, best model (RF) drops 0.647 → 0.639.** The 650M checkpoint is null on *both* framings. This completes the A/B: ESM model size is irrelevant to the honest metric.

1. **The wins are small, the losses are lumpy.** SVM-Linear +0.031, KNN +0.011, MLP +0.009 — all within fold noise. But CatBoost −0.035 and LightGBM −0.012 drop hard, mirroring the 3-class result (CatBoost −0.025). The model that rode the new features hardest in Run VIII (CatBoost) is the most sensitive to the checkpoint swap, which suggests part of its Run VIII edge was tuned to the 150M log-probs.

2. **Best binary model is unchanged in identity, nudged down in value.** Random Forest stays on top (0.647 → 0.639); XGBoost (0.624) and CatBoost (0.637 → 0.601) reorder beneath it. Nothing crosses the ±0.03–0.05 fold-noise band.

3. **Verdict on the "bigger LM" lever: a dead end.** 150M and 650M produce the same GOF/LOF/no-net signal — the zero-shot perturbation score was already saturated at 150M. The remaining levers are unchanged and now the only ones left: **data** (cross-species augmentation at scale), **axis** (effect-vs-no-effect scores like ΔΔG), and **nAChR mechanism** (open/closed conformation, ligand/interface context) for the direction gap that no amount of LM scale touches.

### Run IX vs Run VIII — binary

| Metric | 150M (Run VIII) | 650M (Run IX) | Δ |
|--------|----------------|---------------|---|
| Mean macro F1 (10 models) | 0.604 | 0.603 | −0.000 |
| Best model F1 | 0.647 (Random Forest) | 0.639 (Random Forest) | −0.008 |
| Models improved | — | 6 / 10 | noise |


---

## Run IX — Overall: ESM-2 650M is a null on both tasks

| Framing | CV | Mean Δ F1 | Best (150M → 650M) |
|---|---|---|---|
| 3-class | subunit (honest) | −0.006 | CatBoost 0.508 → RF 0.510 |
| Binary GOF/LOF | subunit (honest) | −0.000 | RF 0.647 → RF 0.639 |

**The single finding.** Scaling ESM-2 from 150M to 650M (4.3× parameters, ~2× slower end-to-end, 2.6 GB weights) changes nothing on the honest gene-grouped metric — 3-class −0.006, binary −0.000. The zero-shot "how much does this perturb the protein?" signal saturates at 150M; the extra parameters refine fine-grained likelihoods that don't affect the coarse functional-direction output.

**What this means for the thesis.** Run VIII's +0.023 (3-class) was a *feature-type* win (conservation + ESM-2), not a *model-capacity* win. Run IX closes the capacity question: bigger LMs are not the lever. The levers that remain are **data volume** (mouse/rat augmentation — the biggest headroom), **axis** (ΔΔG and other effect-vs-no-effect scores for cheap 3-class gains), and **nAChR-specific mechanism** (open/closed conformation, ligand/interface context) — the only signal that can close the GOF/LOF *direction* gap, which generic scores are provably blind to (Run VIII).

**Housekeeping.** Revert `embeddings.py` to `esm2_t30_150M_UR50D` — it's ~4× faster to extract and equally good. Cache note: the feature cache keys on the data hash only, so any future ESM-model swap must `python -m vep_nachr2.training.runner clean` first (already done for this run).


# Run X: + AlphaMissense (71 → 72 features) — null result; confirms direction ⊥ pathogenicity

**Date:** 2026-08-22
**Config:** 717 variants, **72 features**, 10 models, subunit CV (gene-grouped, no leakage), 3 seeds (42/123/456), 30 Optuna trials.
**Change vs Run VIII:** one new feature group — **AlphaMissense** (`alphamissense_pathogenicity`, 1 feature): DeepMind's per-substitution pathogenicity score (Cheng et al. 2023), keyed by UniProt accession. ESM-2 reverted to 150M per Run IX.

## The feature + a CHRNA1 numbering fix

AlphaMissense scores *pathogenicity* (0–1, "how damaging is this substitution?") at each UniProt position. Two implementation notes:

1. **RefSeq→UniProt position map.** AlphaMissense is keyed by the UniProt canonical sequence, but this project uses RefSeq numbering. 15/16 genes align perfectly, but **CHRNA1**'s UniProt entry (P02708, 482 aa) carries a 25-aa P3A alternative-splice exon the RefSeq isoform (NP_000070.1, 457 aa) lacks. A `difflib` alignment builds a per-gene RefSeq→UniProt map (identity for 15 genes, +25 offset after ~pos 78 for CHRNA1), applied before lookup.
2. **Wildtype cross-check.** A variant whose recorded wildtype doesn't match AlphaMissense's wildtype at the (mapped) position is a data-entry error; its score is imputed (in-batch mean) rather than force-mapped.

Final: **687/717 (95.8%) clean matches**, 0 missing; 30 rows (4.2%) are genuine data-entry errors, neutral-imputed.

## Key takeaways

**AlphaMissense is a null on both framings — mean Δ +0.003 (3-class) and +0.006 (binary), both inside fold noise.** This is exactly what Run VIII predicted: AlphaMissense measures *pathogenicity* (damaging vs benign), which is **orthogonal to direction of effect** (GOF vs LOF). A "damaging GOF" and a "damaging LOF" variant get near-identical AlphaMissense scores, so it can't separate them.

**The 1-seed ablation agrees.** Dropping `alphamissense` *improves* RF macro-F1 in both framings (+0.018 3-class, +0.021 binary) — i.e. it is pure noise for this task. The only features that consistently help when dropped are the base physicochemical / substitution / positional groups (negative delta).

**Verdict: remove AlphaMissense.** It earns no F1 and, per the "don't keep a feature that doesn't improve the score" rule, should be dropped from the default pipeline. This is itself a *positive thesis finding*: direction-of-effect is a distinct problem from pathogenicity, and stacking a SOTA pathogenicity score on top of the direction model adds nothing.

**Caveat on the ablation.** It runs **1 seed** (seed 42) for speed, so its per-group deltas are point estimates (±0.01–0.03 noise); the 3-seed compare above is the controlled number for AlphaMissense, and it says null. The same 1-seed pass shows ESM-2 and conservation as slightly *negative* — at odds with Run VIII's 3-seed +0.023 for the pair — so treat those two as "needs a 3-seed re-ablation", not settled.

**3-class:** *mean — Run VIII 0.471 → Run X 0.474, Δ +0.003*

| Model | Run VIII (71) | Run X (72) | Δ F1 |
|---|---|---|---|
| Random Forest | 0.5012 | 0.5028 | +0.002 |
| Logistic Regression | 0.4834 | 0.4995 | +0.016 |
| CatBoost | 0.5081 | 0.4889 | −0.019 |
| LightGBM | 0.4750 | 0.4800 | +0.005 |
| XGBoost | 0.4695 | 0.4717 | +0.002 |
| SVM (Linear) | 0.4745 | 0.4705 | −0.004 |
| KNN | 0.4651 | 0.4657 | +0.001 |
| Gaussian NB | 0.4586 | 0.4657 | +0.007 |
| MLP | 0.4669 | 0.4475 | −0.019 |
| SVM (RBF) | 0.4099 | 0.4462 | +0.036 |

**Binary:** *mean — Run VIII 0.604 → Run X 0.609, Δ +0.006*

| Model | Run VIII (71) | Run X (72) | Δ F1 |
|---|---|---|---|
| XGBoost | 0.6245 | 0.6447 | +0.020 |
| Random Forest | 0.6469 | 0.6304 | −0.017 |
| LightGBM | 0.6169 | 0.6263 | +0.009 |
| Logistic Regression | 0.5978 | 0.6156 | +0.018 |
| CatBoost | 0.6366 | 0.6121 | −0.025 |
| MLP | 0.5955 | 0.6067 | +0.011 |
| SVM (Linear) | 0.5793 | 0.6030 | +0.024 |
| KNN | 0.5858 | 0.5970 | +0.011 |
| Gaussian NB | 0.5853 | 0.5925 | +0.007 |
| SVM (RBF) | 0.5675 | 0.5639 | −0.004 |

**Leave-one-group-out ablation (RF, 1 seed) — negative = helps:**
*Full-model (1-seed) F1 — 3-class 0.4494 | binary 0.6234*

| Condition | 3-class ΔF1 | binary ΔF1 |
|---|---|---|
| drop_positional | −0.063 | +0.010 |
| drop_physicochemical | −0.013 | −0.019 |
| drop_substitution | −0.006 | −0.057 |
| drop_conservation | −0.005 | +0.015 |
| drop_structural_nachr | +0.012 | +0.015 |
| drop_esm2 | +0.013 | +0.026 |
| drop_conformational | +0.014 | +0.015 |
| drop_alphamissense | +0.018 | +0.021 |
| drop_structural_core | +0.034 | −0.053 |


---
## Run XI: 4 mechanism features (82 features) — neutral-to-negative; gene identity is the only signal

**Date:** 2026-08-18
**Config:** 717 variants, **82 features**, subunit CV (gene-grouped, no leakage), Random Forest, 1 seed (42), 5 outer folds, 30 Optuna trials.
**Change vs Run X:** AlphaMissense dropped (Run X null). Four mechanism/structure feature groups added:

| Group | n | What it encodes |
|-------|---|-----------------|
| `stability` | 2 | burial-weighted ΔΔG proxy — `(hyd_wt−hyd_mt)·burial`, `|vol_mt−vol_wt|·burial` (the ESM-IF/ThermoMPNN slot, without the broken torch_geometric dep) |
| `functional_motif` | 5 | flags: in TM / in M2 / in gate ring / in Cys-loop / in agonist-binding aromatic box |
| `charge_context` | 4 | signed charge change × {whole, TM, M2, binding-site} |
| `conformational` | 5 | real α7 open↔closed deltas (RSA, B-factor, interface, pore radius, CA RMSD) — 4/5 were stubbed to 0 |

Total 82 features across 11 groups (was 72 across 10).

> **Why these four.** Run VIII proved the effect-vs-no-effect axis (conservation, ESM-2) lifts 3-class but is blind to GOF↔LOF *direction*. These four target the direction gap directly: where the mutation lands in the receptor (motif, charge-context), what it does to folding (stability), and whether it biases the open/closed equilibrium (conformational). They are the nAChR-mechanism hypothesis made explicit.

*Full model (82 feats) macro F1 = 0.4487. Δ = F1(drop) − F1(full); negative = helps.*

| Condition | ΔF1 (3-class) | Read |
|---|---|---|
| drop_positional | −0.0578 | 🔴 critical — only real signal |
| drop_conservation | −0.0001 | ⚪ neutral |
| drop_structural_core | +0.0002 | ⚪ neutral |
| drop_functional_motif | +0.0038 | 🟢 NEW — neutral |
| drop_physicochemical | +0.0090 | 🟢 (1-seed noise) |
| drop_charge_context | +0.0108 | 🟢 NEW — neutral |
| drop_structural_nachr | +0.0139 | 🟢 neutral |
| drop_conformational | +0.0142 | 🟢 NEW — neutral |
| drop_esm2 | +0.0143 | 🟢 (1-seed noise) |
| drop_stability | +0.0201 | 🟢 NEW — neutral |
| drop_substitution | +0.0239 | 🟢 (1-seed noise) |


### Run XI (3-class ablation) Key Takeaways

**The four new features are all neutral-to-negative on the honest 1-seed RF metric.** Full-model F1 = 0.4487 (≈ Run X's 0.4494 at 1 seed — swapping AlphaMissense out for 11 mechanism features is a wash). Dropping any one of the four *improves* F1 by +0.004…+0.020, i.e. each is, at best, dead weight and, at worst, mild noise.

1. **The only robust signal is gene identity.** `drop_positional` = −0.058 — the largest move by 2.5×, matching every run since Run I. Which subunit a variant sits in is what the model actually learns.

2. **⚠️ Read the small deltas as noise, not ranking.** In this single 1-seed pass, *substitution* (+0.024) and *physicochemical* (+0.009) also show up as "harmful" — directly contradicting Run II's 10-model 3-seed result (−0.017 / −0.016, the #2/#3 features). A 1-seed RF point estimate cannot rank features inside a ±0.02 band. So the honest claim is not "stability is worse than functional_motif" — it's "all four sit at or below zero; none helps."

3. **The mechanism hypothesis did not land.** Run VIII's framing was that direction needs nAChR-mechanism signal (open/closed conformation, ligand/interface context). Four concrete attempts at that signal — motif flags, charge-in-context, burial-weighted stability, real α7 conformational deltas — none moved GOF↔LOF. Hand-crafted receptor-geometry features are too coarse to separate gain- from loss-of-function; the direction signal is not in *where* the residue sits or *how buried* it is.

4. **Remaining levers are unchanged and now the only ones left.** Data volume (cross-species augmentation — the next step) and learned (not hand-crafted) representations. The feature-engineering well is, at this point, dry.

### Verdict

Drop all four new groups from the default pipeline (per the "don't keep a feature that doesn't improve the score" rule). If a single one is worth keeping for thesis narrative, it's `conformational` — the real α7 deltas are scientifically defensible and were already in the feature list (just stubbed) — but it earns no F1 either.

*(Binary GOF/LOF ablation skipped at the user's request.)*

---
## Run XII: fix `structural_nachr` (5 dead features) — full F1 +0.010 (1-seed), `positional` still the only robust signal

**Date:** 2026-08-26 · **Model:** Random Forest · **CV:** nested leave-one-subunit-out (gene-grouped, no leakage), 30 Optuna trials · **Data:** 717 variants, 71 features.

**Change vs Run XI:** reverted to the 71-feature default (Run XI's four mechanism groups dropped), then **fixed two extractors** — no new features added.

1. `structural_nachr` computed 5 of its 7 features wrong (raw UniProt position used as a PDB residue key → pore/interface/burial silently 0.0; binding-site coords never populated; `subunit_burial` was a `return 0.0` placeholder; TM table only 10/16 genes). Rebuilt on `mechanism.py` + the shared alignment maps.

| feature | before | after |
|---|---|---|
| `pore_distance` | ~0 (wrong key) | **91.5%** non-zero |
| `ligand_proximity` | 0 (never populated) | **48.4%** non-zero |
| `interface_proximity` | ~0 (wrong key) | **88.1%** non-zero |
| `interface_contacts` | 0.8% (CA-only cutoff) | **31.4%** non-zero (heavy-atom) |
| `subunit_burial` | constant 0 (placeholder) | **31.4%** non-zero |
| `tm_helix` / `tm_depth` | 10/16 genes | **16/16** genes |

2. `conformational` had `ca_rmsd` ↔ `pore_radius_change` swapped (columns 3↔4). Value-invariant under Random Forest (column order doesn't change a tree ensemble) → **no performance effect**, interpretability only.

| Metric | pre-fix (1-seed) | post-fix (1-seed) | post-fix (5-seed) |
|---|---|---|---|
| Macro F1 (headline) | 0.4569 | 0.4665 | **0.4931 ± 0.025** |
| MCC | 0.2552 | 0.2752 | 0.3032 ± 0.036 |
| Balanced accuracy | 0.5104 | 0.5346 | 0.5214 ± 0.024 |
| Accuracy | 0.5199 | 0.5292 | 0.5628 ± 0.026 |
| Macro precision | 0.4919 | 0.5112 | 0.5195 ± 0.028 |
| Macro recall | 0.4786 | 0.4975 | 0.5140 ± 0.024 |


**Reading the overall numbers.** The fix lifts macro F1 **+0.010** (0.4569 → 0.4665) at 1 seed, and the multi-seed level settles at **0.4931 ± 0.025** — the 5-seed mean is the number to quote, since seed 42's point estimate (0.4665) understates the model. MCC (+0.020) and balanced accuracy (+0.024) move together with F1, so the gain is a real signal, not an accuracy artifact.

Context for the headline metric: with a 363 / 164 / 190 class split, a majority-class (LOF) baseline scores ~0.51 accuracy but only **0.33 balanced accuracy** (one class predicted). The model's 0.52 balanced accuracy is above that chance floor, but the gap to a clinically useful predictor is still wide — that's the whole motivation for adding data, not features.

*Δ = F1(drop group) − F1(full); negative = group helps.*

| group | ΔF1 (1-seed) | ΔF1 (5-seed) | std (5-seed) |
|---|---|---|---|
| positional | −0.0557 | **−0.0124** | 0.041 |
| conservation | −0.0131 | −0.0033 | 0.024 |
| structural_nachr | −0.0118 | −0.0019 | 0.026 |
| esm2 | −0.0066 | +0.0096 | 0.030 |
| substitution | −0.0059 | +0.0009 | 0.030 |
| physicochemical | −0.0012 | +0.0066 | 0.029 |
| structural_core | +0.0034 | +0.0070 | 0.017 |
| conformational | +0.0120 | +0.0071 | 0.019 |

*Full-model macro F1: 1-seed 0.4665 (pre-fix 0.4569, +0.010) | 5-seed 0.4931 ± 0.025*


### Run XII takeaways

1. **The fix did its job, but it's a cleanup, not a new lever.** `structural_nachr` flipped from *harmful* in Run XI (+0.014, dead columns) to neutral/slightly-helpful here (−0.002 ± 0.026). The five zero-columns now carry signal, so the group no longer drags the model. Full F1: **0.4569 → 0.4665** (1-seed) and **0.4931 ± 0.025** (5-seed).

2. **`positional` (gene identity) is the only robust signal — again.** −0.012 is the largest delta and the only one that clears its own std. This has held across every run: the model is effectively "which subunit is it" plus weak residue-level signal.

3. **Conformational: DROP.** `drop_conformational` = +0.007 (removing it *helps*) at 5 seeds, +0.012 at 1 seed, and it is CHRNA7-only (5 columns = 0.0 for ~93% of variants). Removed from the default pipeline → **66 features / 7 groups**. Kept as an importable module for the thesis ablation section.

4. **The 1-seed ↔ 5-seed flip is the point of the confirmation.** `substitution`, `physicochemical`, `structural_core`, `esm2` all change sign between 1 and 5 seeds — that's the ±0.02–0.03 noise band, not feature ranking. Only `positional` (reliably negative) and `conformational` (structural case for dropping) are settled; the rest need data, not more features.

---
## Run XIII: "new ideas" screen — embeddings, stacking, ordinal, LoRA all null or negative

**Date:** 2026-09-02 · **CV:** nested leave-one-subunit-out (gene-grouped, no leakage), 30 Optuna trials, 1 seed (42) · **Data:** 760 multi-species variants (human 507 / rat 173 / mouse 80; LOF 377 / NNE 188 / GOF 195), 66 baseline vs 82 full features.

**Purpose:** after Run XII declared the hand-crafted feature well dry, this screen tests the four *learned / architectural* ideas in one pass. Every idea is compared against the 66-feature baseline on identical data and seed, so the deltas below are clean A/Bs.

| # | Idea | Result (macro F1) | Verdict |
|---|---|---|---|
| #1 | LoRA fine-tune ESM-2 150M (r=8) | **0.267** | below the 0.33 chance floor — strong negative |
| #2 | Stability ΔΔG via RaSP | — | BLOCKED (not pip-installable) |
| #3 | Per-residue ESM-2 embeddings (+16 PCA) | Δ ≈ +0.001 | null |
| #4 | Ensemble stacking (logistic meta-learner) | 0.463 | below catboost 0.490 — null |
| #5 | Ordinal regression (cumulative-link) | 0.378 baseline | worst of six — firm negative |


*Best baseline: catboost 0.4900 · stacking = 0.4629 · LoRA = 0.2668 (below chance)*

| Model | baseline | +embeddings | Δ F1 |
|---|---|---|---|
| catboost | 0.4900 | 0.4750 | −0.015 |
| lightgbm | 0.4759 | 0.4384 | −0.038 |
| logistic_regression | 0.4556 | 0.4677 | +0.012 |
| random_forest | 0.4328 | 0.4318 | −0.001 |
| xgboost | 0.4103 | 0.4236 | +0.013 |
| ordinal | 0.3783 | 0.4118 | +0.034 |


**Reading the numbers.** Each idea is an *addition* to an already-strong baseline (catboost 0.490, in the same band as Run XII's 0.493 on 717 human variants — different data, so treat as a level, not an exact compare). None clears it:

- **Embeddings (#3) — a wash.** Mean Δ +0.001, but the sign is model-class-dependent: the linear models gain (ordinal +0.034, LR +0.012) while the tree models that actually win lose (lightgbm −0.038, catboost −0.015). The 16 PCA components are noise the trees overfit, and they only lift models too weak to matter.
- **Ordinal (#5) — firm negative.** 0.378 baseline is the worst of all six by a wide margin; the `LOF < NNE < GOF` ordering assumption is actively wrong. NNE (188 variants) sits ambiguously between the two functional extremes, so forcing a monotone link discards signal.
- **Stacking (#4) — does not beat the best single model.** 0.463 < catboost 0.490; the meta-learner averages the weaker models down instead of extracting a complement.
- **LoRA (#1) — the clearest negative.** 0.267 is below the 0.33 chance floor: fine-tuning a 150M-param ESM-2 on 760 variants (grouped by gene) is too little data, and it overfits worse than classical ML. Strongest evidence yet that learned representations cannot rescue a dataset this small.


### Run XIII takeaways

1. **Every remaining non-data idea is ruled out.** Embeddings (null), stacking (null), ordinal (negative), LoRA (strongly negative), RaSP (blocked). Combined with Run XII's four neutral-to-negative mechanism groups, the feature and architecture space is exhausted.

2. **The ceiling is the data, not the model.** catboost 0.490 (1-seed, 760 multi-species variants) is the honest number, and no lever that avoids new data moved it.

3. **Learned representations are orthogonal to direction, at every scale.** Zero-shot ESM-2 scores (Run VIII), per-residue embeddings (here), and full LoRA fine-tuning (here) all fail to separate GOF↔LOF — reaffirming that direction-of-effect is a distinct problem from pathogenicity, and that 760 labels is far below what even parameter-efficient transfer needs.

4. **The next step is more curated data, not more models.** The 760 variants already span human/rat/mouse, so the open lever is expanding the curated variant count (datascraper/ + human-automation pipelines), not fancier features or architectures.


## Run XIV: fix structural position→PDB mapping + drop `structural_nachr`

The "structural features don't help" question had a concrete answer: the position→PDB
mapping was broken, so `structural_core` was reading features from the *wrong residues*.
Fixing it removes the harm but — consistent with Run XIII — does not turn structure into
a strong direction signal. `structural_nachr` is dropped from the default set (sparse, neutral).


### What was broken

1. **5 genes homology-mapped to a different subunit** — CHRNA2/5/6, CHRNB3, CHRNG were mapped to
   another gene's chain (~47% identity), so every structural feature came from the wrong residue.
2. **CHRNA10 used the wrong isoform** — AF-Q13002 (908 aa) instead of the canonical Q9GZZ6 (450 aa).
3. **The 7 AlphaFold models had no `.dssp`** — RSA fell back to a fill value (1.0) for every position.

### Fix

- Gapped global alignment (BioPython `PairwiseAligner`, BLOSUM62, open −10 / extend −0.5) +
  **<80% identity rejection** in `structural.py::_build_alignment_map`.
- Repointed 6 genes to their **own AlphaFold model** in `config.py::PDB_MAPPING`.
- Real solvent accessibility via **Shrake-Rupley SASA** in `structural.py::_compute_dssp_dict_from_structure`
  (bypasses `mkdssp`, not installed).

Result: all 760 variants map to real RSA (91.6% overall; the previously-broken genes 100%).


*Δ = F1(drop) − F1(full); full F1 = 0.4715 (75-feat, RF 1-seed). structural_core: +0.007 (broken) → −0.002 (fixed).*

| Condition | n_feat | macro F1 | ΔF1 |
|---|---|---|---|
| drop_positional | 55 | 0.4047 | −0.0668 |
| drop_substitution | 72 | 0.4388 | −0.0327 |
| drop_esm_embeddings | 59 | 0.4559 | −0.0156 |
| drop_structural_core | 68 | 0.4691 | −0.0024 |
| drop_esm2 | 73 | 0.4711 | −0.0004 |
| full | 75 | 0.4715 | 0.0000 |
| drop_physicochemical | 51 | 0.4731 | +0.0015 |
| drop_conservation | 72 | 0.4756 | +0.0041 |


### Run XIV takeaways

1. **The mapping bug is fixed; structure is now *neutral*, not harmful.** `structural_core` flipped
   from +0.007 (broken mapping → wrong-residue features) to −0.002 (fixed → correct residues +
   real RSA). Fixing the bug removed the drag but did not create a positive signal — structure is
   still not a direction lever, consistent with Run XIII.

2. **The earlier 82-feat run overstated the structural win.** In that run `structural_core` showed
   −0.019, but that was redundancy with the (harmful) `structural_nachr` group. Once `structural_nachr`
   is dropped, `structural_core` reverts to −0.002. Read per-group deltas with the *full* feature set in mind.

3. **`structural_nachr` removed from the default set.** Its pore/interface/burial features are 0.0 for the
   7 AlphaFold-monomer genes (single chain → no interface) and its ligand feature is α-only
   (~72% of the group is zeros). Dropping it lifts full F1 0.4597 → 0.4715 and simplifies 82 → 75
   features. Module kept for the thesis ablation section.

4. **The ceiling is unchanged: data.** This is a correctness cleanup, not a new lever — the headline
   number is still catboost 0.490 (Run XIII). The structural feature space is now clean (correct mapping,
   real ASA, no degenerate group); the open lever remains curated-data expansion.


## Run XV: structural overhaul (mutation-aware features) — NEGATIVE, reverted

Run XIV fixed the position→PDB mapping so `structural_core` reads the *right* residue,
but it stayed neutral (−0.002). Run XV attacks the question head-on: diagnose every reason
structure is weak, fix each one, and re-measure.


### The 5 reasons structural features were weak, and the fixes

1. **WT-only features.** `structural_core` (RSA, Cβ-density, B-factor, DSSP, is_unmappable)
   describes the *wild-type* residue, so every substitution at a position gets an identical
   value → no mutation signal. **Fix:** add mutation-aware extractors.
2. **B-factor scale mixing.** pLDDT (AlphaFold) vs crystallographic B (experimental PDBs) are
   different scales, blended into one column. **Fix:** z-score within each structure type.
3. **`is_unmappable` is ~92% dead.** Only ~8% of variants are unmappable → a near-constant column.
4. **Missing nAChR biology.** No M2-gate / Cys-loop / binding-pocket / TM membership.
   **Fix:** FunctionalMotifExtractor — 5 binary flags at the GOF/LOF functional sites.
5. **No burial × chemical-change interaction.** A buried Ala→Val matters more than a surface
   one. **Fix:** StabilityExtractor — burial × Δhydrophobicity and burial × Δvolume (a
   structure-aware ΔΔG proxy).

**Three extractors wired in:** StabilityExtractor (2), FunctionalMotifExtractor (5),
ChargeContextExtractor (4 — Δcharge × location). Plus structural_core expanded to 9 features
(pLDDT + contact_number + z-scored B-factor) → 88-feat "full overhaul".


**Random Forest (82-feat, full F1 0.4577):**

| Condition | n_feat | macro F1 | ΔF1 |
|---|---|---|---|
| drop_positional | 62 | 0.4213 | −0.0364 |
| drop_esm2 | 80 | 0.4294 | −0.0282 |
| drop_stability | 80 | 0.4562 | −0.0015 |
| full | 82 | 0.4577 | 0.0000 |
| drop_functional_motif | 77 | 0.4598 | +0.0022 |
| drop_substitution | 79 | 0.4625 | +0.0048 |
| drop_conservation | 79 | 0.4666 | +0.0089 |
| drop_esm_embeddings | 66 | 0.4670 | +0.0093 |
| drop_physicochemical | 58 | 0.4687 | +0.0110 |
| drop_structural_core | 75 | 0.4742 | +0.0165 |

**CatBoost (82-feat, full F1 0.4468):**

| Condition | n_feat | macro F1 | ΔF1 |
|---|---|---|---|
| drop_positional | 62 | 0.4270 | −0.0198 |
| drop_esm_embeddings | 66 | 0.4442 | −0.0026 |
| full | 82 | 0.4468 | 0.0000 |
| drop_stability | 80 | 0.4511 | +0.0043 |
| drop_esm2 | 80 | 0.4542 | +0.0074 |
| drop_substitution | 79 | 0.4580 | +0.0113 |
| drop_conservation | 79 | 0.4605 | +0.0138 |
| drop_physicochemical | 58 | 0.4702 | +0.0234 |
| drop_functional_motif | 77 | 0.4802 | +0.0334 |
| drop_structural_core | 75 | 0.4840 | +0.0373 |

**Headline (1-seed):** 75-feat base RF 0.4715 / CB 0.4773 (kept) · 88-feat full 0.4581 / 0.4672 · 82-feat refined 0.4577 / 0.4468


### Run XV takeaways

1. **The structural overhaul did not flip it — it regressed it.** Every mutation-aware /
   structure-expanded config scored *below* the 75-feat baseline on both models. The
   "helpful" ablation deltas for stability/functional_motif were ~±0.02 noise: they flipped
   sign between the 88-feat and 82-feat runs (1 seed → macro_f1_std = 0.0).

2. **Conclusion: single-residue structural context carries no direction-of-effect signal for
   nAChR GOF/LOF — WT-side or mutation-aware.** The 5 diagnosed reasons were real and fixed,
   yet none of the fixes produced a robust lift. The signal is *absent*, not *broken*.

3. **Repo reverted to the 75-feat default set** (Run XIV baseline, catboost 0.490). The three
   mutation-aware modules (stability.py, functional_motif.py, charge_context.py) are kept on
   disk for the thesis ablation section but are NOT in the default set.

4. **Where the direction signal actually lives** (consistent with Run XIII): sequence-level
   conservation + ESM-2 embeddings + positional one-hot. Structure is not a lever; the
   remaining headroom is the data (curated variants + label quality), not features.


## Run XVI: α7-only structural diagnostic — the professor's two-state hypothesis, tested

Runs XII–XV settled that single-residue structural context is neutral/negative in the **full
16-gene model** — where most genes lack a dedicated structure and the receptor is pentameric.
The professor's counter-proposal: *run CHRNA7 (α7) alone* — it's **homomeric** (no interface
ambiguity), the most-represented gene, and the **only one with both a closed (7EKT) and open
(7KOX) experimental structure** — and if the two-state signal works there, later "map other
genes to α7 by approximation."

This run tests that hypothesis directly, in the one gene where complete two-state structure
has the best possible chance to carry signal.


### Setup

- **Data:** `load_mutation_data(subunits=["CHRNA7"])` → 200 variants (human 119 / rat 80 /
  mouse 1). Labels: LOF 100 / No-net-effect 58 / GOF 42.
- **Features:** 75-feat default **+ ConformationalExtractor** (the α7-unique open↔closed
  delta group, NOT in the default set) = 80 features. The 16 dead `subunit_*` one-hots are
  masked (constant for a single gene) → 64 features.
- **CV:** `nested_cross_validation(cv_mode="standard")` — plain StratifiedKFold (gene grouping
  is meaningless for one gene), **3 seeds** so `macro_f1_std` is a real confidence interval,
  not the 1-seed `std=0` that plagued Runs XIV/XV.
- **Confound control:** the `species_*` columns carry a human↔rat↔label shortcut (human-α7 is
  84 LOF / 16 GOF; rat-α7 is 39 NNE / 25 GOF), so a `no_species` run masks them out.


**CatBoost α7-only (full F1 0.6015 ± 0.0257, 3 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| drop_physicochemical | 40 | 0.5715 | 0.0066 | −0.0300 |
| drop_positional | 60 | 0.5743 | 0.0138 | −0.0272 |
| no_species | 61 | 0.5796 | 0.0084 | −0.0220 |
| drop_structural_core | 57 | 0.5865 | 0.0278 | −0.0150 |
| drop_esm_embeddings | 48 | 0.5916 | 0.0068 | −0.0099 |
| drop_substitution | 61 | 0.5937 | 0.0016 | −0.0078 |
| drop_esm2 | 62 | 0.5959 | 0.0158 | −0.0056 |
| drop_conformational | 59 | 0.5979 | 0.0414 | −0.0036 |
| full | 64 | 0.6015 | 0.0257 | 0.0000 |
| drop_conservation | 61 | 0.6106 | 0.0227 | +0.0091 |

**RandomForest α7-only (full F1 0.5821 ± 0.0072, 3 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| drop_conformational | 59 | 0.5692 | 0.0292 | −0.0128 |
| full | 64 | 0.5821 | 0.0072 | 0.0000 |
| no_species | 61 | 0.5849 | 0.0063 | +0.0029 |
| drop_structural_core | 57 | 0.5931 | 0.0169 | +0.0110 |

**open↔closed conformational delta (drop_conformational − full):** CatBoost −0.004 (±0.041) · RandomForest −0.013 (±0.029)


### Run XVI takeaways — the hypothesis is settled

1. **α7 alone "works" at the surface level: macro F1 ≈ 0.60 / 0.58** (catboost / RF). But this
   is *not* evidence for structure — it's the single-gene + species shortcut inflating the score
   (a 3-class task on 200 variants with a human-84-LOF / rat-39-NNE split is an easier, partially
   species-leakable problem than the full 16-gene task).

2. **The open↔closed conformational group is null even in α7.** `drop_conformational` = −0.004
   (catboost, ±0.041) and −0.013 (RF, ±0.029). The std is 3–10× the effect in both models. In the
   *one* homomeric gene with *complete* two-state experimental structure (7EKT closed → 7KOX open),
   the conformational delta does not predict GOF/LOF/NNE direction. This is the direct, clean test
   of the professor's hypothesis — and it fails.

3. **structural_core is weak and sign-flips** (−0.015 catboost vs +0.011 RF) — the same noise we
   saw in Runs XIII–XV, now confirmed with real seed variance in the most structure-rich gene.

4. **The species confound is real but model-dependent** (`no_species` = −0.022 catboost vs
   +0.003 RF): catboost exploits the human↔rat↔label shortcut more than RF does. Either way it
   does not dominate — physicochemical (−0.030) is the strongest lever, and that is *sequence*
   chemistry, not structure.

**Bottom line for the professor:** the direction signal is genuinely *absent* from single-residue
structure, even in the ideal homomeric two-state case. The "map other genes to α7 by approximation"
path is therefore moot — mapping to a structure that carries no signal cannot create one. The lever
is data (more curated variants, label quality), not structure. The conformational/structural modules
stay on disk for the thesis ablation section but remain out of the default set.


## Run XVII: H1 functional-region partition — NEGATIVE

Runs XIII–XVI showed single-residue *geometry* (RSA/DSSP/B-factor/conformational) carries no
direction signal. H1 is the next hypothesis up the abstraction ladder: **GOF/LOF direction is a
function of *which functional element* the residue sits in**, not its static geometry. This is
the nAChR analogue of ENaC's spatially-partitioned signal (Liddle GOF in the C-terminal PY motif,
LOF in the pore/ECD) — so if the analogy holds, a coarse *region* label should separate direction
where the fine-grained geometry could not.

Implemented as `FunctionalRegionExtractor` — 7 mutually-exclusive one-hot regions
(`agonist_binding_site`, `membrane_proximal_ecd`, `extracellular_other`, `m2_m3_linker`,
`pore_m2`, `other_tmd`, `intracellular_loop`) from the TM boundaries + binding-site residues in
`mechanism.py` (annotation-only, no PDB). Three conditions:
1. `baseline` (75 feats) · 2. `baseline_plus_region` (82, the primary test) ·
3. `baseline_swap` (−structural_core +region, still 75 — is region a *better* use of 7 columns?).


### Setup + the reason H1 looked promising

- **Data:** full multi-species 3-class set (760 variants: human 507 / rat 173 / mouse 80;
  LOF 377 / GOF 195 / NNE 188), `cv_mode="subunit"` (honest gene-grouped), **5 seeds**.
- **The smoking gun (pooled crosstab, region × effect):** M2-M3 linker 13 GOF / 2 LOF; pore M2
  39 GOF / 20 LOF; agonist-binding site 11 LOF / 1 GOF. Pooled, the region partition separates
  direction *hard*. The question is whether that survives holding out whole genes.


**RandomForest (baseline F1 0.4849 ± 0.0372, 5 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| baseline | 75 | 0.4849 | 0.0372 | — |
| baseline_plus_region | 82 | 0.4799 | 0.0295 | −0.0050 |
| baseline_swap | 75 | 0.4835 | 0.0340 | −0.0014 |

**CatBoost (baseline F1 0.4915 ± 0.0239, 5 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| baseline | 75 | 0.4915 | 0.0239 | — |
| baseline_plus_region | 82 | 0.4742 | 0.0280 | −0.0172 |
| baseline_swap | 75 | 0.4945 | 0.0310 | +0.0031 |


### Run XVII takeaways — the functional-region hypothesis fails

1. **The region partition does NOT help.** `baseline_plus_region` = −0.005 (RF) / −0.017
   (catboost). `baseline_swap` (−geometry +region) is neutral: −0.001 (RF) / +0.003 (catboost).
   Adding 7 region columns is at best a wash, at worst a small hurt (catboost is the model most
   sensitive to the new columns and it drops most).

2. **Why the strong crosstab vanished under CV:** the pooled region×effect table is a gene/species
   confound in disguise. Variants in a given region are dominated by a *few genes* (e.g. most M2-M3
   linker variants are one or two genes), and gene-grouped CV holds those genes out entirely — so a
   "region → direction" rule that is really a "gene → direction" rule cannot transfer to unseen
   genes. Region membership is also largely redundant with the *positional* group (gene one-hot +
   normalized position) the baseline already has, so the model gets no new signal, only noise.

3. **This closes the structural/functional line of attack.** Runs XIII (embeddings/stacking/ordinal),
   XV (mutation-aware structural), XVI (α7 two-state conformational), and now XVII (functional region)
   are all null or negative under honest gene-grouped CV. The direction-of-effect signal is not in
   single-residue structure *or* coarse functional-region annotation — consistent with the running
   conclusion that the lever is **data** (curated variants, label quality), not better per-residue
   features.

**Verdict:** H1 rejected. `FunctionalRegionExtractor` stays on disk (`scripts/`,
`results/functional_region/`) for the thesis's negative-result/ablation section but is not added to
the default feature set.


## Run XVIII: H2 allosteric-network position — NEGATIVE

H1 (functional-region one-hot, Run XVII) failed because a coarse region label is really a gene/species
proxy in disguise. H2 goes finer and more mechanistic: **GOF/LOF direction is a function of a residue's
position in the allosteric communication network** — its contact-graph centrality and whether it lies on
the pocket→gate transduction path — rather than static geometry or a coarse region tag. The idea: gating
is a global, cooperative conformational change, so a mutation's sign of effect should depend on where the
residue sits along the ligand-binding → pore pathway.

Implemented as `AllostericNetworkExtractor` — 4 features from a residue-residue contact graph
(Cβ–Cβ < 8 Å, Cα for Gly) over all chains of the mapped PDB:
1. `betweenness` (allosteric-hub score) · 2. `closeness` (global centrality) ·
3. `gate_proximity` = 1/(1+d) to the nearest M2 gate-ring residue ·
4. `on_allosteric_path` (on a pocket→gate shortest path).
Three conditions: `baseline` (75), `baseline_plus_allosteric` (79), `baseline_swap` (−geometry +network, 72).


### Setup + the pooled signal that motivated H2

- **Data:** full multi-species 3-class set (760 variants), `cv_mode="subunit"` (honest gene-grouped), **5 seeds**.
- **Coverage:** 696/760 (91.6%) variants map to a residue in the contact graph.
- **The pooled signal:** gate_proximity by effect = GOF 0.263 > LOF 0.156 > NNE 0.130 — GOF variants sit
  systematically closer to the gate in network terms. Sanity: α7 L250T/V251T (spontaneous openers) read
  gate_proximity 1.0, binding-site variants read on_path 1.0. The question is whether this survives
  holding out whole genes.


**RandomForest (baseline F1 0.4849 ± 0.0372, 5 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| baseline | 75 | 0.4849 | 0.0372 | — |
| baseline_plus_allosteric | 79 | 0.4768 | 0.0294 | −0.0081 |
| baseline_swap | 72 | 0.4811 | 0.0329 | −0.0038 |

**CatBoost (baseline F1 0.4915 ± 0.0239, 5 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| baseline | 75 | 0.4915 | 0.0239 | — |
| baseline_plus_allosteric | 79 | 0.4805 | 0.0401 | −0.0110 |
| baseline_swap | 72 | 0.4966 | 0.0300 | +0.0051 |


### Run XVIII takeaways — the allosteric-network hypothesis fails

1. **Adding the network features hurts.** `baseline_plus_allosteric` = −0.008 (RF) / −0.011 (catboost).
   Swapping geometry for network (`baseline_swap`) is a wash: −0.004 (RF) / +0.005 (catboost).
2. **Why the pooled gate_proximity signal vanished** is the same story as H1: gate proximity is dominated
   by pore-M2 variants, which cluster in a few genes, and gene-grouped CV holds those genes out — so a
   "closer-to-gate → GOF" rule that is really a "gene → direction" rule cannot transfer. The 4 columns add
   noise on top of the positional group (gene one-hot + normalized position) the baseline already has.
3. This is the latest in a long run of nulls on per-residue feature engineering (XIII embeddings/stacking/
   ordinal, XV mutation-aware structural, XVI α7 two-state geometry, XVII functional region). Network
   position — the most mechanistically-motivated per-residue feature yet — still does not separate
   direction.

**Verdict:** H2 rejected. `AllostericNetworkExtractor` stays on disk (`scripts/`, `results/allosteric/`)
for the thesis negative-result/ablation section but is not added to the default feature set.


## Run XIX: H3 open↔closed ΔΔG (energy two-state) — NEGATIVE

Run XVI tested the α7 two-state idea with *geometry* (open↔closed RSA/B-factor/interface deltas) and
found nothing. H3 re-asks the same question with *energy*: does a variant's inverse-folding ΔΔG
difference between the OPEN (7KOX) and CLOSED (7EKT) states predict GOF/LOF/NNE? The intuition: a GOF
mutation should be one the open state accommodates better than the closed state does, a LOF the reverse.

Implemented as `OpenClosedDDGExtractor` — ESM-IF (inverse folding) log-likelihood scored against both
structures, 3 features: `ddg_open`, `ddg_closed`, `ddg_diff` (= ddg_open − ddg_closed; negative =
open-favouring). α7-only (the one gene with a complete open+closed pair).


### Setup + the tail signal that motivated H3

- **Data:** α7-only 3-class (200 variants: LOF 100 / GOF 42 / NNE 58), `cv_mode="standard"` (1 gene), **5 seeds**.
- **Coverage:** 197/200 (98.5%) variants score in both states (ESM-IF backbone covers the full α7 chain).
- **The tail signal:** the 8 most open-favouring variants are all GOF (ddg_diff −10.5 to −2.3); the 8 most
  closed-favouring are all LOF (+2 to +8). The mean is noisy — most mutations are ~equally tolerated in
  both states (ddg_diff ≈ 0), so the signal lives in the extremes, not the bulk.


**RandomForest (full(+ddg) F1 0.5885 ± 0.0200, 5 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| full | 62 | 0.5885 | 0.0200 | — |
| drop_ddg | 59 | 0.5751 | 0.0131 | −0.0133 |
| no_species | 59 | 0.5736 | 0.0137 | −0.0149 |

**CatBoost (full(+ddg) F1 0.6001 ± 0.0228, 5 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| full | 62 | 0.6001 | 0.0228 | — |
| drop_ddg | 59 | 0.6028 | 0.0161 | +0.0026 |
| no_species | 59 | 0.5909 | 0.0119 | −0.0093 |


### Run XIX takeaways — the energy two-state test fails too

1. **ΔΔG does not robustly help.** `drop_ddg` delta = −0.013 (RF, a small gain) but +0.003 (catboost, a
   small hurt). Both are within ~1 seed-std (0.013–0.023), so the sign flips across models — noise, not
   signal.
2. **Why the clean tail signal doesn't transfer:** only a handful of variants have |ddg_diff| ≫ 0 (the
   outliers where ESM-IF is confident open-vs-closed differs), and those few carry the pooled direction
   signal. Most mutations are ~equally tolerated in both states, so ddg_diff ≈ 0 for the bulk and a
   threshold on it classifies only a few outliers. Tree models can latch onto those extremes (RF's +0.013),
   but it doesn't generalise — catboost finds nothing.
3. This closes the two-state line of attack from BOTH directions — geometry (XVI) and energy (XIX). The
   direction-of-effect signal is not in single-residue structure, functional-region annotation, network
   position, or open↔closed state preference. The lever remains **data** (curated variants, label quality).

**Verdict:** H3 rejected. `OpenClosedDDGExtractor` stays on disk (`scripts/`, `results/ddg/`) for the
thesis negative-result/ablation section but is not added to the default feature set.


## Run XX: H4 coevolution (ESM-2 attention coupling to gate) — NEGATIVE

H4 asks whether residues that **coevolve with the gating hinge** are the ones whose mutation has a strong
*directional* effect (GOF or LOF) rather than none. Instead of an explicit MSA / EVcouplings run we use
ESM-2's learned attention as a coevolution proxy — the model was trained to predict each residue from the
rest of the sequence, so its attention patterns encode which positions are jointly informative (statistical
coupling). For each gene we run the WT sequence once, take the LAST layer's attention (averaged over heads),
symmetrise into a coupling matrix `C = (A + Aᵀ)/2`, and read off two per-position features:

1. `coev_gate` — mean coupling of the position to the M2 gate-ring residues (middle 40% of TM2).
2. `coev_hub`  — total coupling (column-sum of `C` over real residues), a coevolutionary "centrality".

Full multi-species 3-class set (760 variants), honest subunit-grouped CV (`cv_mode="subunit"`), **5 seeds**,
RF + catboost.


### Setup + the gate-ring sanity check

- **Data:** full multi-species 3-class set (760 variants), `cv_mode="subunit"`, **5 seeds**.
- **Sanity (the extractor works):** the top-10 by `coev_gate` are genuine gate-ring residues — α7 **L250T**
  (the classic spontaneous-opener), CHRNA1 253/254, CHRNB4 272, CHRNA5 261. So the attention coupling to M2
  really is measuring the gating hinge, not noise.
- **The pooled signal (flat):** `coev_gate` by effect = 0.0021 / 0.0019 / 0.0022 (LOF/NNE/GOF) and
  `coev_hub` = 1.011 / 0.995 / 0.996 — indistinguishable. No bulk separation to begin with.


**RandomForest (baseline F1 0.4849 ± 0.0372, 5 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| baseline | 75 | 0.4849 | 0.0372 | — |
| baseline_plus_coev | 77 | 0.4840 | 0.0285 | −0.0009 |

**CatBoost (baseline F1 0.4915 ± 0.0239, 5 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| baseline | 75 | 0.4915 | 0.0239 | — |
| baseline_plus_coev | 77 | 0.4904 | 0.0228 | −0.0010 |


### Run XX takeaways — coevolution coupling is null

1. **Adding the coupling features does nothing.** `baseline_plus_coev` = −0.0009 (RF) / −0.001 (catboost) —
   exactly baseline within noise. The pooled signal was already flat (`coev_gate` 0.0021/0.0019/0.0022), so
   there was nothing for the model to pick up.
2. **Why:** ESM-2 attention coupling to the gate ring does not separate GOF/LOF/NNE. The gate-ring residues
   themselves are the most coupled (as they should be), but that coupling does not rank-order the *effect
   direction* of mutations at those positions. H4 joins the other position/network features — H1 functional
   region, H2 allosteric network — in failing to carry direction signal under gene-grouped CV.
3. This is the **8th consecutive null**.

**Verdict:** H4 rejected. `CoevolutionExtractor` stays on disk (`scripts/`, `results/coev/`) for the thesis
negative-result/ablation section but is not added to the default feature set.


## Run XXI: H5 mutant-minus-WT embedding delta — NEGATIVE

Run XIII used ESM-2 WT embeddings as *position* features (16 PCA comps) and was null. H5 tests a different
quantity: the **difference** between the mutant and WT embeddings at the variant position —

    delta = emb(seq[mt])[pos] − emb(seq[wt])[pos]     (640-dim)

PCA-reduced to 16 components. The idea: the magnitude and direction of this learned-representation shift
should encode how "loaded" a substitution is — whether it nudges the protein toward a functionally different
state (GOF vs LOF vs no change) — orthogonal to the scalar MLM zero-shot score already in `esm2`.

Full multi-species 3-class set (760 variants), honest subunit-grouped CV (`cv_mode="subunit"`), **5 seeds**,
RF + catboost.


### Setup + the flat-magnitude signal

- **Data:** full multi-species 3-class set (760 variants), `cv_mode="subunit"`, **5 seeds**.
- **Coverage:** 100% — every variant gets a delta (WT embedding cached per gene, mutant sequence run per
  variant, then PCA).
- **The pooled signal (flat):** |delta| (L2 norm of the 16 PCs) = 3.21 / 3.28 / 3.23 (LOF/NNE/GOF). The
  mutation's representation shift does NOT scale with effect direction.


**RandomForest (baseline F1 0.4849 ± 0.0372, 5 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| baseline | 75 | 0.4849 | 0.0372 | — |
| baseline_plus_embdelta | 91 | 0.4850 | 0.0195 | +0.00005 |

**CatBoost (baseline F1 0.4915 ± 0.0239, 5 seeds):**

| Condition | n_feat | macro F1 | std | ΔF1 |
|---|---|---|---|---|
| baseline | 75 | 0.4915 | 0.0239 | — |
| baseline_plus_embdelta | 91 | 0.4805 | 0.0160 | −0.0109 |


### Run XXI takeaways — the embedding delta is null

1. **No help.** `baseline_plus_embdelta` = 0.0000 (RF) / −0.0109 (catboost). RF is a dead tie; catboost shows
   a small hurt that is still within ~1 seed-std (0.016), so it is noise, not signal.
2. **Why:** the magnitude of the MT−WT embedding shift is flat across LOF/NNE/GOF (3.21/3.28/3.23), and the 16
   PC directions add no separable information beyond the scalar zero-shot score already in the baseline. In
   other words, *how far* a mutation moves the ESM-2 representation does not tell you *which direction* its
   effect points.
3. This is the **9th consecutive null** on per-variant/per-residue feature engineering.

**Verdict:** H5 rejected. `EmbeddingDeltaExtractor` stays on disk (`scripts/`, `results/emb_delta/`) for the
thesis negative-result/ablation section but is not added to the default feature set.


## Run XXII: subgroup/heterogeneity diagnostic — weak positive (1-seed hint)

Nine consecutive structural hypotheses (Runs XIII, XV–XXI) came back null under pooled
leave-one-subunit-out CV. The working explanation: direction is gene/mechanism-conditioned, and the honest
CV forbids the gene identity that structure would need to be informative. XXII tests the one remaining
version of "structure should help": does `structural_core` carry signal **within** a receptor family,
where the mechanism is homogeneous?

Three subgroups (16 genes, partitioned by assembly): **muscle** (CHRNA1/B1/D/E/G, 5 genes),
**neuronal-heteromeric** (CHRNA2/3/4/5/6/9/10 + CHRNB2/3/4, 10 genes), **neuronal-homomeric** (α7, 1 gene).
Per subgroup: full vs `drop_structural_core` ablation, `cv_mode="subunit"` (leave-one-gene-out) where ≥2
genes, `standard` for α7 (single gene — ceiling with position leakage, see Run XVI). RF + CatBoost, 1 seed,
15 Optuna trials.


### Setup + the within-family question

- **Data:** full multi-species 3-class set (760 variants), 1 seed, 15 trials, RF + CatBoost.
- **Metric:** `delta_f1 = F1(drop_structural_core) − F1(full)`; **negative = structural_core helps**.
- **Permutation importance** of the 7 structural columns per family (descriptive single 80/20 split, not CV).


**Δ = F1(drop_structural_core) − F1(full); negative = structural_core helps:**

| Subgroup | Model | CV | full F1 | ΔF1 |
|---|---|---|---|---|
| muscle | RF | subunit/5 | 0.4003 | −0.0132 |
| muscle | CatBoost | subunit/5 | 0.4725 | −0.0125 |
| neuronal_hetero | RF | subunit/5 | 0.4095 | −0.0097 |
| neuronal_hetero | CatBoost | subunit/5 | 0.4502 | **−0.0469** |
| neuronal_homo (α7) | RF | standard/5 | 0.5375 | −0.0020 |
| neuronal_homo (α7) | CatBoost | standard/5 | 0.5739 | −0.0107 |

**structural_core permutation importance (descriptive, single split):**

| feature | muscle | neuronal_hetero | neuronal_homo |
|---|---|---|---|
| rsa | −0.005 | −0.010 | 0.015 |
| cbeta_density | −0.003 | −0.003 | −0.002 |
| b_factor | **0.045** | 0.000 | 0.006 |
| dssp_helix | −0.005 | −0.015 | −0.004 |
| dssp_sheet | 0.015 | −0.009 | 0.005 |
| dssp_coil | −0.001 | −0.015 | 0.007 |
| is_unmappable | 0.000 | 0.000 | 0.000 |


### Run XXII takeaways — a weak within-family signal, not a clean null

1. **structural_core helps within every family** (all deltas negative): muscle −0.013 (RF) / −0.012
   (catboost), neuronal-hetero −0.010 / **−0.047** (catboost), α7 −0.002 / −0.011. The pooled delta
   (Run XII / `ablate_feature_impact`) was ≈ −0.002 — effectively zero. Structure is *more* informative
   within a family than across all 16 genes, exactly as the mechanism-conditioning story predicts.
2. **But it is 1 seed and not corroborated by permutation importance.** No single structural column rises
   above noise except `b_factor` in muscle (0.045) and `rsa` in α7 (0.015); neuronal-hetero columns are all
   ≤ 0. So the ablation delta is either (a) real but distributed across columns via weak interactions, or
   (b) 1-seed noise. The catboost neuronal-hetero −0.047 is the one number worth a confirmation run.
3. **α7 is the ceiling, not the lever.** Within-α7 F1 is 0.54–0.57 (standard CV, position leakage) — the
   same ceiling Run XVI found. Structure barely moves it (−0.002 / −0.011).

**Verdict:** a **weak positive** — structure is not *dead*, it is *under-powered*: it adds a small,
family-local signal that the pooled 16-gene CV washes out. The family-conditioned model (Run XXIV) is the
right follow-up: if direction is genuinely more predictable within a family, that is where structure's
contribution shows up.


## Run XXIII: label-quality audit — NNE is noisy, but it doesn't move the metric

The 3-class metric inherits every label error in `final_mapped.xlsx`. XXIII audits label provenance, flags
the rows most likely to be mislabeled, and then tests whether the lowest-confidence labels actually change
the score. No new model — pandas + the existing ablation path.

**Key finding up front:** "No net effect" (NNE) is the weakest-evidence class — 186/194 (95.9%)
single-source and 78/194 (40%) contradicted by AlphaMissense (pathogenicity > 0.564) — yet dropping those
78 rows does **not** help the score.


### Setup + label provenance

- **Source:** `data/raw/final_mapped.xlsx` (841 rows: LOF 420 / GOF 213 / NNE 194 / LOF-GOF 14).
- **Provenance columns:** `n_papers` (source count), `n_rows_merged`, `mapping_status` (position-mapping
  confidence), `Correct?` (manual review). AlphaMissense scores via `AlphamissenseExtractor`.
- **Relabel sensitivity:** drop the 78 NNE rows AlphaMissense calls pathogenic, re-run 3-class CV
  (RF, 1 seed, subunit-grouped), compare F1.


**Effect × provenance:**

| Effect | n | single-source % | mean papers |
|---|---|---|---|
| LOF | 420 | 92.6% | 1.09 |
| GOF | 213 | 88.7% | 1.15 |
| No net effect | 194 | **95.9%** | 1.06 |
| LOF/GOF | 14 | 100% | 1.00 |

**Flag reasons (n):** nne_single_source 186 · low_conf_mapping 78 · nne_am_pathogenic 78 · multi_row_merged 71 · wt_mismatch 30 · ambiguous_effect 14 · manual_review 7

**Relabel sensitivity (drop 78 AM-pathogenic NNE rows):**

| Condition | n | macro F1 | MCC | balanced acc |
|---|---|---|---|---|
| full | 760 | 0.4558 | 0.2910 | 0.4913 |
| drop_am_pathogenic_nne | 682 | 0.4486 | 0.2597 | 0.4872 |


### Run XXIII takeaways — the NNE noise is real but orthogonal

1. **NNE is the most single-source class.** 186/194 NNE labels (95.9%) rest on one paper, vs 92.6% LOF and
   88.7% GOF. "No effect" is the weakest conclusion, so this is expected — but it means the middle class is
   the least reliable.
2. **40% of NNE is contradicted by AlphaMissense.** 78/194 NNE rows have AM pathogenicity > 0.564 — a
   strong pathogenicity model calls them likely-damaging. On its face that looks like mislabeling.
3. **But it doesn't matter for the direction task.** Dropping those 78 rows moves F1 0.4557 → 0.4486
   (−0.007, within noise). This is the *same* result as Run VIII: pathogenicity (AlphaMissense) is orthogonal
   to direction-of-effect. An AM-pathogenic "no effect" is not a hidden LOF/GOF label from this model's
   point of view — it is a genuinely different axis.
4. 342/841 rows flag for manual review (nne_single_source 186, low_conf_mapping 78, multi_row_merged 71,
   wt_mismatch 30, ambiguous_effect 14, manual_review 7) — a concrete checklist for the thesis data section.

**Verdict:** label noise exists and is concentrated in NNE, but it is **orthogonal** to GOF/LOF direction
and does not explain the metric ceiling. No relabeling step is justified by the data.


## Run XXIV: family-conditioned CV — negative (family doesn't help)

Run XXII found `structural_core` helps *within* every receptor family (all deltas negative), and framed
that as "structure is under-powered, not dead — the pooled 16-gene CV washes out a family-local signal."
The natural follow-up: if direction is mechanism-conditioned and family ≈ mechanism, a model that only
ever sees one family should reach a higher macro-F1 than the pooled 16-gene model. XXIV tests exactly
that, with the honest leave-one-gene-out CV inside each family.

Three conditions: **muscle** (CHRNA1/B1/D/E/G, 5 genes, n=257), **neuronal-heteromeric** (CHRNA2/3/4/5/6/
9/10 + CHRNB2/3/4, 10 genes, n=303), and **pooled** (all 16 genes, n=760) as the reference. RF + CatBoost,
3 seeds, 15 Optuna trials, `cv_mode="subunit"` (leave-one-gene-out, capped at 5).


### Setup + the family-conditioning question

- **Data:** full multi-species 3-class set (760 variants); each family is a row-mask, no re-extraction.
- **Metric:** macro-F1; within-family vs pooled delta (positive = family helps).
- **Caveat:** families are smaller than pooled (257/303 vs 760), so within-family F1 is noisier and the
  two are not sample-matched — interpret directionally, not as a precise effect size.


| Family | Model | CV | n | genes | macro F1 | std | MCC | bal acc |
|---|---|---|---|---|---|---|---|---|
| muscle | RF | subunit/5 | 257 | 5 | 0.4123 | 0.0090 | 0.2777 | 0.4938 |
| muscle | CatBoost | subunit/5 | 257 | 5 | 0.4642 | 0.0068 | 0.2879 | 0.5311 |
| neuronal_hetero | RF | subunit/5 | 303 | 10 | 0.3934 | 0.0559 | 0.1740 | 0.4507 |
| neuronal_hetero | CatBoost | subunit/5 | 303 | 10 | 0.4247 | 0.0898 | 0.1865 | 0.4671 |
| **pooled** | RF | subunit/5 | 760 | 16 | **0.5030** | 0.0343 | 0.3193 | 0.5271 |
| **pooled** | CatBoost | subunit/5 | 760 | 16 | **0.5044** | 0.0211 | 0.3072 | 0.5271 |

*Within-family vs pooled Δ (positive = family helps): RF muscle −0.091 / hetero −0.110 · CatBoost muscle −0.040 / hetero −0.080.*


### Run XXIV takeaways — family-conditioning strictly underperforms

1. **Family-conditioning lowers F1, everywhere.** Muscle −0.091 (RF) / −0.040 (catboost);
   neuronal-heteromeric −0.110 / −0.080, against pooled 0.503 / 0.504. Both families, both models, every
   delta negative. The hypothesis predicted the *opposite* — a family-local model should do better — so
   this is a clean refutation, not a null.
2. **The pooled model's gene one-hot + cross-family data is doing real work.** Pooled sees all 16 genes
   from 760 rows; within-family throws away both (257/303 rows). That dropping to one family *hurts*
   means cross-family data is a net positive: direction transfers something useful across genes.
3. **Closes the XXII loop.** XXII's weak within-family structural signal (1-seed) looked like a
   family-local effect worth chasing. XXIV shows the within-family *baseline* is itself weaker than
   pooled — so even if `structural_core` genuinely helped within a family, it would be helping a worse
   model. There is no family-conditioned ceiling to reach.

**Verdict:** **negative.** The last structural escape hatch — slicing the data by receptor family — is
closed. Ten hypotheses, ten negatives: single-residue structure does not encode GOF/LOF direction for
nAChR, and neither does conditioning on receptor family. The ~0.50 pooled macro-F1 is a real ceiling for
this feature set, not an artifact of heterogeneity.


## Run XXV: GNM allosteric-dynamics features — null (11th)

Ten structural hypotheses (Runs XIII, XV–XXIV) were null under honest leave-one-subunit-out CV. The
literature pointed at one thing we had never actually tried: **protein dynamics**. PreMode (Zhong et al.,
Nat. Comms. 2025) found GoF variants enrich at the inner-helix gate but states that static structure is
insufficient for ion channels and dynamics is the missing signal. Ersoy et al. (eLife 2023) computed
per-residue **allosteric coupling to the CFTR gate** with a Gaussian Network Model and found the coupling
hotspots overlap disease mutations.

Run XXV implements exactly that for nAChR — a `GatingExtractor` (`vep_nachr2/features/gating.py`) that
computes three **Gaussian Network Model (GNM)** features on the Cα backbone:

- `gnm_msf` — theoretical mean-square fluctuation (thermal flexibility)
- `gnm_slowmode` — participation in the 2 slowest collective (hinge/gating) modes
- `gnm_gate_coupling` — cross-correlation of the residue to the channel gate (the M2 9′ constriction ring;
  0 for the 7 AlphaFold-monomer genes that have no assembled pore)

Prior runs used static geometry, boolean gate membership, static contact graphs, spatial gate proximity,
end-state ΔΔG, and an ESM-2 *attention* proxy — but never the actual mechanical fluctuation correlation.
This is the additive ablation: baseline 75 vs 75+gating (78), pooled 16 genes, RF + CatBoost, 3 seeds, 15
Optuna trials.


### Setup + the "dynamics is missing" hypothesis

- **Data:** full multi-species 3-class set (760 variants). Additive ablation (baseline vs baseline+gating).
- **Metric:** macro-F1; delta = +gating − baseline (positive = gating helps).
- **GNM:** Kirchhoff matrix on Cα (7.3 Å cutoff), covariance = Γ⁺; gate = Cα residues within 4 Å of the
  pore-constriction minimum radial distance (principal-axis method).


| Condition | Model | n_feat | macro F1 | std | MCC | bal acc |
|---|---|---|---|---|---|---|
| baseline | RF | 75 | 0.5030 | 0.0343 | 0.3193 | 0.5271 |
| baseline | CatBoost | 75 | 0.5044 | 0.0211 | 0.3072 | 0.5271 |
| baseline_gating | RF | 78 | 0.5081 | 0.0205 | 0.3273 | 0.5301 |
| baseline_gating | CatBoost | 78 | 0.4994 | 0.0396 | 0.3115 | 0.5225 |

*Δ (+gating − baseline): RF +0.005 · CatBoost −0.005.*


### Run XXV takeaways — dynamics is null too; the ceiling is real

1. **Opposite signs = null.** RF +0.005 (0.5030 → 0.5081), CatBoost −0.005 (0.5044 → 0.4994). Both are
   inside the ±0.02–0.03 cross-seed std. MCC drifts the same tiny amount (+0.008 RF / +0.004 CatBoost) —
   noise, not effect.
2. **This closes the last literature-grounded escape hatch.** Every structural flavor is now ruled out:
   static geometry, boolean region membership, contact graphs, spatial gate proximity, end-state ΔΔG,
   language-model attention, and now true mechanical (GNM) allosteric coupling. The one signal PreMode
   explicitly says static structure lacks — dynamics — is also null here.
3. **Why it's null (mechanism, not measurement):** the M2 9′ gate *is* the GOF hotspot (Leu9′→polar =
   slow-channel GOF), but the *direction* of a gate-region mutation is gene/subunit-conditioned — the
   same coupling value means GOF in one subunit and LOF in another. Leave-one-subunit-out CV forbids gene
   identity, so a per-residue coupling feature cannot disambiguate direction. The descriptive pooled
   gradient (GOF 0.059 > LOF 0.054 > NNE 0.042) is the same gene confound seen in Runs XVII–XVIII.

**Verdict: null (11th consecutive).** Structure — static *or* dynamic — does not encode GOF/LOF direction
for nAChR gene-agnostically. The ~0.50 pooled macro-F1 is a robust ceiling for single-residue features;
direction is mechanism- and gene-conditioned, and cannot be recovered without gene identity.


## Run XXVI: gene-aware / mechanism-conditioned model — POSITIVE (+0.13)

Twelve structural/heterogeneity hypotheses were null under leave-one-subunit-out CV (Runs XIII, XV–XXV),
and the ~0.50 pooled macro-F1 looked like a hard ceiling. But that metric forbids the one thing a
clinician always knows: **the gene**. This run asks a different, defensible question — *given the gene,
how well can we predict GOF/LOF/NNE direction?* — and lets gene identity be a legitimate input.

Four conditions decompose the answer (full multi-species 3-class set, 760 variants; RF + CatBoost; 10
seeds × 15 Optuna trials; `scripts/run_xxvi_gene_aware.py`):

- **gene_prior** — analytic majority-class-per-gene baseline (no model). The floor: direction signal in
  gene identity alone.
- **gene_agnostic** — 75 features, `cv_mode="subunit"` (leave-one-gene-out). The honest predictor.
- **gene_aware** — 75 features, `cv_mode="standard"` (stratified K-fold: gene identity is a legitimate
  test-time input). THE gene-aware model.
- **variants_only** — 55 features (drop positional: gene OH + species OH + norm_pos), `cv_mode="standard"`.
  Per-variant biochemistry in-distribution, to isolate whether explicit gene identity is what carries the
  gain.


### Setup + the "direction is gene-conditioned" question

- **CV semantics:** `standard` = plain StratifiedKFold (a gene's variants appear in both train and test —
  the in-distribution ceiling, and the legitimate diagnostic question). `subunit` = StratifiedGroupKFold
  by gene (generalization to unseen genes — the honest baseline).
- **Metric:** macro-F1 (3-class). Deltas: gene_aware − gene_agnostic = value of knowing the gene;
  gene_aware − gene_prior = per-variant contribution within a known gene; gene_aware − variants_only =
  marginal value of explicit gene identity.


### Results (10 seeds × 15 trials)

| Condition | Model | n_feat | macro F1 | std | MCC | bal acc |
|---|---|---|---|---|---|---|
| gene_prior | majority_class | 1 | 0.4069 | 0.0000 | 0.2420 | 0.4437 |
| gene_agnostic | RF | 75 | 0.4781 | 0.0371 | 0.2809 | 0.4992 |
| gene_agnostic | CatBoost | 75 | 0.4887 | 0.0403 | 0.2895 | 0.5095 |
| **gene_aware** | RF | 75 | **0.6359** | 0.0072 | 0.4613 | 0.6370 |
| **gene_aware** | CatBoost | 75 | **0.6401** | 0.0078 | 0.4682 | 0.6454 |
| variants_only | RF | 55 | 0.5967 | 0.0134 | 0.4054 | 0.5995 |
| variants_only | CatBoost | 55 | 0.6001 | 0.0129 | 0.4120 | 0.6052 |

*Δ vs gene_agnostic: gene_aware **+0.158** (RF) / **+0.151** (CB) · variants_only +0.119 / +0.111 ·
gene_aware vs gene_prior +0.229 / +0.233.*

#### Conditions

| Item | Value |
|---|---|
| New thing | Treat gene identity as a legitimate model input and evaluate under gene-aware `standard` CV (the prior ~0.50 ceiling was a gene-agnostic CV artifact, not a data limit). |
| Dataset | 760 variants (507 human, 173 rat, 80 mouse) |
| Classes | LOF 377 / No-net-effect 188 / GOF 195 |
| Genes | 16 subunits (CHRNA1-7, A9, A10, CHRNB1-4, CHRND, CHRNE, CHRNG) |
| Features | 75 (gene_agnostic + gene_aware) · 55 (variants_only, drops 20 positional cols) |
| Seeds | 10 (matched per-seed across conditions) |
| Optuna trials | 15 per inner fold |
| CV | gene_agnostic: `subunit` (leave-one-gene-out) · gene_aware + variants_only: `standard` (5×5 StratifiedKFold) |
| Models | Random Forest, CatBoost |
| Metric | macro-F1 (3-class), reported as per-seed mean ± std |

#### Significance (paired t-test over 10 seeds)

| Comparison | RF Δ | RF p | CB Δ | CB p |
|---|---|---|---|---|
| gene_aware > gene_agnostic | +0.158 | < 0.0001 | +0.151 | < 0.0001 |
| gene_aware > variants_only | +0.039 | < 0.0001 | +0.040 | < 0.0001 |
| variants_only > gene_agnostic | +0.119 | < 0.0001 | +0.111 | < 0.0001 |
| gene_aware > gene_prior (0.4069) | +0.229 | < 0.0001 | +0.233 | < 0.0001 |


### Run XXVI takeaways — direction is predictable given the gene

1. **Knowing the gene is worth ~+0.15.** gene_aware (0.636 RF / 0.640 CatBoost) vs gene_agnostic
   (0.478 / 0.489), paired t-test p < 0.0001 for both models. The 10-seed estimate of gene_agnostic is
   actually *lower* than the earlier 3-seed number (~0.50 → ~0.48), so the gene-aware gain is now
   larger, not smaller. The 12-run "~0.50 ceiling" was a gene-agnostic artifact, not a data limit.
2. **It is not just the gene prior.** Majority-class-per-gene is 0.407; the full model reaches ~0.64.
   Per-variant features add ~+0.23 *within* a known gene (biochemistry + ESM embeddings + structure do
   real work on top of the prior), p < 0.0001 (one-sample vs prior).
3. **Explicit gene identity is worth only ~+0.04 on top of in-distribution biochemistry.** variants_only
   (0.597/0.600) already beats gene_agnostic by +0.12 (p < 0.0001), because per-variant features
   (position-relative, conservation, ESM embeddings) implicitly encode gene-typical patterns that
   transfer within a gene. The one-hot itself adds the last +0.04. The gain is mostly *being
   in-distribution*, not the identity feature per se.
4. **Variance collapses in the gene-aware setting.** gene_aware std 0.007/0.008 vs gene_agnostic
   0.037/0.040 — leave-one-gene-out is inherently high-variance; the gene-aware estimate is tight.

**Verdict: positive (confirmed at 10 seeds).** A nAChR GOF/LOF predictor is genuinely feasible
(≈0.64 macro-F1) when gene identity is an input — the correct framing for a diagnostic tool. The honest
generalization-to-unseen-genes number remains ≈0.48; the practical, gene-conditioned predictor is the
deliverable to carry forward.


## Run XXVII: allosteric coupling × mutation — POSITIVE under gene-aware CV

A colleague's "indirect connection" idea, run under the new gene-aware protocol. Hypothesis: a
mutation's effect is transmitted to the functional machinery through coupled residues, so the
*interaction* of allosteric coupling with the mutation's chemical change should carry direction signal
that neither quantity alone does.

Runs XVIII (contact-graph allosteric network) and XXV (GNM gate coupling) tested raw gate proximity and
raw GNM coupling — both gene-AGNOSTIC null. This run re-tests the raw GNM gating group *and* adds a new
`AllostericInteractionExtractor` (`vep_nachr2/features/allosteric_interaction.py`) — the product of gate
coupling and mutation chemistry — under gene-aware CV (`cv_mode="standard"`).

Three conditions, 3-class, RF + CatBoost, 10 seeds × 15 trials:

- **baseline** (75)
- **baseline_gating** (78) — re-test Run XXV's gnm_msf / gnm_slowmode / gnm_gate_coupling
- **baseline_allosteric** (79) — allosteric_dvol / dhydro / dcharge / dpolarity (coupling × Δproperty)


### Setup + the "indirect connection" hypothesis

- **Interaction = coupling × chemistry.** gate_coupling ∈ [0,1] (|mean correlation| to the M2 9′ gate;
  0 for the 7 AlphaFold-monomer genes) times Δproperty ∈ [−1,1] (variant − wildtype AAIndex change).
  Signed: a big volume *increase* at a gate-coupled residue scores large-positive; any change at an
  uncoupled residue ~0.
- **Metric:** macro-F1; delta = condition − baseline (positive = the group helps).
- **Protocol:** gene-aware (standard CV) per the project's new default — gene identity is a legitimate
  input; direction is gene-conditioned, so "coupled to the gate" can mean GOF in one subunit and LOF in
  another, which only a gene-aware model can learn.


### Results (10 seeds × 15 trials)

| Condition | Model | n_feat | macro F1 | std | MCC | bal acc |
|---|---|---|---|---|---|---|
| baseline | RF | 75 | 0.6359 | 0.0072 | 0.4613 | 0.6370 |
| baseline | CatBoost | 75 | 0.6401 | 0.0078 | 0.4682 | 0.6454 |
| **baseline_gating** | RF | 78 | **0.6480** | 0.0152 | 0.4794 | 0.6496 |
| **baseline_gating** | CatBoost | 78 | **0.6523** | 0.0141 | 0.4863 | 0.6568 |
| baseline_allosteric | RF | 79 | 0.6367 | 0.0122 | 0.4625 | 0.6380 |
| baseline_allosteric | CatBoost | 79 | 0.6431 | 0.0101 | 0.4721 | 0.6482 |

*Δ vs baseline: gating **+0.012** (RF) / **+0.012** (CB) · allosteric +0.001 / +0.003.*

#### Conditions

| Item | Value |
|---|---|
| New thing | Add allosteric-interaction features (gate coupling × mutation Δchemistry) and re-test the GNM gating group under gene-aware CV. |
| Dataset | 760 variants (507 human, 173 rat, 80 mouse) |
| Classes | LOF 377 / No-net-effect 188 / GOF 195 |
| Genes | 16 subunits (CHRNA1-7, A9, A10, CHRNB1-4, CHRND, CHRNE, CHRNG) |
| Features | 75 (baseline) · 78 (baseline_gating: +gnm_msf / gnm_slowmode / gnm_gate_coupling) · 79 (baseline_allosteric: +allosteric_dvol / dhydro / dcharge / dpolarity = coupling × Δproperty) |
| Seeds | 10 (matched per-seed across conditions) |
| Optuna trials | 15 per inner fold |
| CV | `standard` (5×5 StratifiedKFold, gene-aware) for all conditions |
| Models | Random Forest, CatBoost |
| Metric | macro-F1 (3-class), per-seed mean ± std |

#### Significance (paired t-test over 10 seeds)

| Comparison | RF Δ | RF p | CB Δ | CB p |
|---|---|---|---|---|
| baseline_gating > baseline | +0.012 | 0.011 | +0.012 | 0.007 |
| baseline_allosteric > baseline | +0.001 | 0.422 (ns) | +0.003 | 0.058 (ns) |


### Run XXVII takeaways — gating is a small real win; the interaction term is null

1. **Gating is positive and now statistically significant.** +0.012 (RF, p = 0.011) / +0.012 (CatBoost,
   p = 0.007). This is the first structural feature to clear the significance bar under gene-aware CV.
   But the magnitude shrank from the 3-seed headline (+0.019/+0.013) — it's a small, consistent lift.
2. **The coupling×chemistry interaction is null.** allosteric +0.001 (RF, p = 0.42) / +0.003 (CatBoost,
   p = 0.058). The 3-seed "+0.008/+0.005" was noise. Raw GNM fluctuation/coupling carries the signal;
   the explicit product term does not. This matches Run XXVIII, which found the interaction redundant
   with gating.
3. **The gene-aware reframe is what exposes it.** Run XXV's *exact* GNM features were null gene-agnostic
   (opposite signs, +0.005/−0.005). Under gene-aware CV they're positive *and* significant — structure's
   contribution only appears once the model can use gene identity.
4. **Magnitude is modest.** +0.012 is real but small; the bigger lever remains more per-gene data, not
   more structural features.

**Verdict: positive but narrower than the 3-seed headline** — gating is a significant +0.012; the
allosteric interaction term is null. Carry-forward: gene-aware + gating (0.648–0.652).


## Run XXVIII: per-feature structural ablation — only GNM flexibility (slowmode) survives

Run XXVII treated the structural features as whole groups. This run drops each of the 14 structural
columns **one at a time** (and re-drops the 3 whole groups as anchors) against a full 82-feature
reference (base 75 + gating 3 + allosteric 4), all under gene-aware CV (`cv_mode="standard"`), 3-class,
RF + CatBoost, **10 seeds × 15 trials**.

Delta = (drop feature) − (full); **negative = the feature helps**.


### Setup — 14 structural columns, 3 group anchors

- **Reference:** full 82-feat set (**0.6451 RF / 0.6470 CatBoost** — the carry-forward gene-aware + gating +
  interaction model, a hair above Run XXVII's +gating, within noise).
- **Per-feature drops:** structural_core {rsa, cbeta_density, b_factor, dssp_helix, dssp_sheet, dssp_coil,
  is_unmappable}, gating {gnm_msf, gnm_slowmode, gnm_gate_coupling}, allosteric {dvol, dhydro, dcharge,
  dpolarity}.
- **Group anchors:** drop whole structural_core / gating / allosteric, to read each feature against its group.
- **Noise floor:** full std ≈ ±0.010–0.013; individual-feature deltas of ±0.005–0.009 are ~1σ, so only
  `gnm_slowmode` clears significance — treat everything else as within run-to-run noise.


*Reference: full 82-feat = 0.6451 (RF) / 0.6470 (CatBoost). Δ = F1(drop) − F1(full); negative = feature helps.
\* = paired p < 0.05 (RF only — nothing is significant for CatBoost).*

| dropped | RF Δ | CatBoost Δ |
|---|---|---|
| drop_group_structural_core | −0.0017 | −0.0059 |
| drop_rsa | −0.0032 | −0.0011 |
| drop_cbeta_density | −0.0032 | −0.0028 |
| drop_b_factor | −0.0002 | −0.0003 |
| drop_dssp_helix | −0.0015 | −0.0005 |
| drop_dssp_sheet | −0.0045 | −0.0013 |
| drop_dssp_coil | −0.0032 | +0.0013 |
| drop_is_unmappable | −0.0023 | −0.0017 |
| drop_gnm_msf | −0.0014 | −0.0039 |
| drop_gnm_slowmode | **−0.0089\*** | −0.0061 |
| drop_gnm_gate_coupling | −0.0008 | +0.0026 |
| drop_group_gating | −0.0084 | −0.0040 |
| drop_allosteric_dvol | +0.0002 | +0.0005 |
| drop_allosteric_dhydro | +0.0002 | −0.0038 |
| drop_allosteric_dcharge | +0.0021 | +0.0008 |
| drop_allosteric_dpolarity | +0.0027 | +0.0006 |
| drop_group_allosteric | +0.0028 | +0.0052 |

*(raw: `results/structural_feature_ablation/structural_feature_ablation_summary.csv`)*


### Run XXVIII takeaways — the only load-bearing structural feature is GNM flexibility (slowmode)

1. **The gating signal survives, but narrows to `gnm_slowmode` alone.** Dropping slowmode costs
   **RF −0.0089 (p = 0.028)** — the *only* individual structural feature that clears significance.
   `gnm_msf` (−0.0014) and `gnm_gate_coupling` (−0.0008 / +0.0026) are individually null. So the signal
   is **global residue flexibility** (the slowest normal mode), not "how connected you are to the M2 9′
   gate ring." The mechanistic story from Run XVIII/XXV flips: *mobility* matters, *connectivity* doesn't.
2. **The allosteric interaction terms are redundant — dropping them *helps*.** drop_group_allosteric =
   +0.003 (RF) / +0.005 (CatBoost), and every individual term is ≤ ±0.004. Run XXVII's coupling×Δchemistry
   gain disappears once gating is in the reference. The "indirect connection" hypothesis closes as a clean
   negative.
3. **`structural_core` is neutral for *both* models — a 10-seed downgrade.** The 3-seed run claimed
   "RF leans on structural_core (−0.014)"; at 10 seeds it is −0.002 (RF) / −0.006 (CatBoost), both ns.
   The earlier "RF-specific structural signal" was seed noise.
4. **`is_unmappable` is no longer doing work.** 3-seed −0.012/−0.007 → 10-seed −0.002/−0.002. The worry
   that the data-quality flag was a "gene identity in a structural costume" is much weaker once averaged
   over 10 seeds — it carries nothing on its own here.

**Verdict: structure carries almost no direction signal.** After the long string of structural nulls
(Runs XIII–XXI, XXV), the 10-seed per-feature sweep leaves a single robust survivor — GNM slowmode
(flexibility). Carry forward the full 82-feat gene-aware model (**0.6451 / 0.6470**), but do not
over-claim a structural mechanism; the real lever remains more per-gene data (Runs XXIX/XXX).


## Run XXIX: species ablation — cross-species data clearly helps (monotonic +0.05)

Does pooling mouse/rat with human actually add signal, or just label noise? The user asked whether to add
more species; this tests whether the cross-species data we ALREADY have is paying for itself. Gene-aware
(`cv_mode="standard"`), 3-class, RF + CatBoost, 10 seeds × 15 trials, full 82-feat set (base 75 + gating 3
+ allosteric 4). Delta vs human_only (positive = the added species helps).


### Setup — 3 training sets, one known reference

- **human_only** (507) — the target distribution alone.
- **human_mouse** (587) — does mouse add signal?
- **human_rat** (680) — does rat add signal?
- **all_three** (760) — already on disk (Run XXVIII full = 0.6451 / 0.6470); not re-run.
- **Label distribution shifts matter:** human data is LOF-heavy (293 LOF / 137 GOF / 77 NNE). Mouse adds
  ~25 NNE / 25 GOF; rat is NNE-rich (+86 NNE, ~50% of its 173 rows). The under-represented classes are
  exactly what cross-species data fills in.


### Results (10 seeds × 15 trials)

| Condition | Model | n | macro F1 | std | MCC | bal acc |
|---|---|---|---|---|---|---|
| human_only | RF | 507 | 0.6021 | 0.0088 | 0.4285 | 0.5926 |
| human_only | CatBoost | 507 | 0.5971 | 0.0102 | 0.4181 | 0.5964 |
| human_mouse | RF | 587 | 0.6216 | 0.0074 | 0.4494 | 0.6185 |
| human_mouse | CatBoost | 587 | 0.6195 | 0.0137 | 0.4514 | 0.6242 |
| human_rat | RF | 680 | 0.6297 | 0.0054 | 0.4511 | 0.6307 |
| human_rat | CatBoost | 680 | 0.6380 | 0.0122 | 0.4675 | 0.6441 |

*Δ vs human_only: +mouse **+0.020** / **+0.022** · +rat **+0.028** / **+0.041**. all_three (760, 82-feat) = Run XXVIII full = 0.6451 (RF) / 0.6470 (CatBoost).*

#### Conditions

| Item | Value |
|---|---|
| New thing | Cross-species augmentation — add mouse/rat rows to the human-only set and measure the per-species gain. |
| Dataset | human_only 507 · human_mouse 587 · human_rat 680 · all_three 760 = 0.6451/0.6470 |
| Classes | LOF 377 / No-net-effect 188 / GOF 195 (full); subsets as above |
| Features | 82 (base 75 + gating 3 + allosteric 4) |
| Seeds | 10 (matched per-seed across conditions) |
| Optuna trials | 15 per inner fold |
| CV | `standard` (5×5 StratifiedKFold, gene-aware) |
| Models | Random Forest, CatBoost |
| Metric | macro-F1 (3-class), per-seed mean ± std |

#### Significance (paired t-test over 10 seeds)

| Comparison | RF Δ | RF p | CB Δ | CB p |
|---|---|---|---|---|
| human_mouse > human_only | +0.020 | 0.0001 | +0.022 | 0.001 |
| human_rat > human_only | +0.028 | < 0.0001 | +0.041 | < 0.0001 |
| human_rat > human_mouse | +0.008 | 0.017 | +0.019 | 0.007 |


### Run XXIX takeaways — cross-species augmentation is real, and it works by rebalancing the labels

1. **Cross-species data clearly helps — monotonically and significantly.** human_only 0.602 (RF) / 0.597
   (CatBoost) → +mouse +0.020/+0.022 (p ≤ 0.001) → +rat +0.028/+0.041 (p < 0.0001). The monotonic step
   (human_rat > human_mouse) is itself significant (p = 0.017 / 0.007). Every species addition is
   positive in both models, now with p-values, not just ~4σ eyeballing. Answer to "will more species
   help?": **yes, cross-species data pays for itself.**
2. **The gain is class rebalancing, not just volume.** Human clinical data is LOF-heavy (293 LOF vs 137
   GOF vs only 77 NNE) because neutral/"no net effect" variants rarely get reported clinically — they
   surface in systematic mutagenesis screens, which live in rodents. Mouse adds ~25 NNE/25 GOF; rat adds
   ~86 NNE (50% of rat rows are NNE). The model's hardest class is exactly the one cross-species data
   fills in.
3. **This sharpens the "add more species?" decision.** The lift comes from (a) more per-gene data and
   (b) NNE/GOF rebalancing — NOT from species diversity per se. So the highest-value next step is more
   **mouse** (under-represented at 80 vs rat 173) and more **rat** (NNE-rich), not a brand-new distant
   species that would need its own alignment and transfer assumption.
4. **Consistency with the gene-aware story.** Under gene-aware CV, adding a species = adding rows
   *per gene*, which directly attacks the per-gene sparsity that was the real limit. This validates the
   project's "principled cross-species augmentation" secondary contribution (agnology / functional
   equivalence): cross-species pooling works, but it should be weighted by functional equivalence, not
   dumped naively.

**Verdict: positive and decisive (confirmed at 10 seeds).** Cross-species augmentation is a real win
(+0.03–0.04 over human-only, all significant), driven by label rebalancing and per-gene density. The
mouse/rat path is worth continuing; a new distant species is lower priority than getting more of the
species we already align.


## Run XXX: data-size & class-balance ablation — data IS the anchor (steep, no plateau)

Is the model limited by the *amount* of data, the *balance* of classes, or is it at an intrinsic ceiling?
Two factors varied independently, gene-aware (`cv_mode="standard"`), 3-class, RF + CatBoost, 10 seeds × 15
trials, full 82-feat set. The full 760 anchor (= Run XXVIII reference) is filled in once XXVIII lands.

- **SIZE** at fixed ~50/25/25 balance: quarter (~196), half (~385), full (760, on disk).
- **BALANCE** at fixed N≈350: 50/25/25 random, 33/33/33 random, 33/33/33 gene-stratified.

Full results, conditions, and p-values below.


### Setup — size vs balance, and a "better idea" for the 33/33/33 selection

- **Per-gene fractional sampling** (quarter/half): each (gene, class) group is scaled by the fraction, so
  gene coverage AND the ~50/25/25 balance are both preserved — the fair way to shrink a gene-aware dataset.
- **Random stratified** (n350_502525, n350_333333): exact per-class counts, global random.
- **Gene-stratified 33/33/33** (the "better idea"): a *random* 350-sample can drop an entire rare gene's
  variants, which is exactly the information the gene-aware model leans on (gene identity = #1 feature).
  This variant samples each class proportionally per gene, so every gene stays represented.
- **Noise caveat:** quarter/half std is large (±0.03–0.04 for RF) — small-sample variance.


### Results (10 seeds × 15 trials)

| Condition | Model | n | macro F1 | std | MCC | bal acc |
|---|---|---|---|---|---|---|
| quarter (50/25/25) | RF | 196 | 0.5566 | 0.0284 | 0.3625 | 0.5627 |
| quarter (50/25/25) | CatBoost | 196 | 0.5386 | 0.0247 | 0.3362 | 0.5471 |
| half (50/25/25) | RF | 385 | 0.5744 | 0.0210 | 0.3784 | 0.5763 |
| half (50/25/25) | CatBoost | 385 | 0.5725 | 0.0147 | 0.3765 | 0.5789 |
| full (50/25/25) | RF | 760 | 0.6451 | 0.0096 | 0.4746 | 0.6468 |
| full (50/25/25) | CatBoost | 760 | 0.6470 | 0.0129 | 0.4782 | 0.6530 |
| random 50/25/25 | RF | 350 | 0.5878 | 0.0109 | 0.3964 | 0.5888 |
| random 50/25/25 | CatBoost | 350 | 0.5844 | 0.0201 | 0.3902 | 0.5897 |
| random 33/33/33 | RF | 350 | 0.6130 | 0.0128 | 0.4251 | 0.6143 |
| random 33/33/33 | CatBoost | 350 | 0.6015 | 0.0237 | 0.4082 | 0.6035 |
| gene-stratified 33/33/33 | RF | 348 | 0.6056 | 0.0096 | 0.4163 | 0.6081 |
| gene-stratified 33/33/33 | CatBoost | 348 | 0.6074 | 0.0120 | 0.4168 | 0.6093 |

*"full" = 760 rows at 50/25/25 = the Run XXVIII 82-feat reference (0.6451 / 0.6470).*

#### Conditions

| Item | Value |
|---|---|
| New thing | Quantify how macro-F1 scales with training size and class balance (subsample N; rebalance 50/25/25 vs 33/33/33). |
| Dataset | subsets of the 760-variant set: quarter 196 · half 385 · three N≈350 draws (full 760 = anchor) |
| Classes | 3-class LOF / No-net-effect / GOF; sampled at 50/25/25 or 33/33/33 |
| Features | 82 (base 75 + gating 3 + allosteric 4) |
| Seeds | 10 (matched per-seed across conditions) |
| Optuna trials | 15 per inner fold |
| CV | `standard` (5×5 StratifiedKFold, gene-aware) |
| Models | Random Forest, CatBoost |
| Metric | macro-F1 (3-class), per-seed mean ± std |

#### Significance (paired t-test over 10 seeds, within-run comparisons)

| Comparison | RF Δ | RF p | CB Δ | CB p |
|---|---|---|---|---|
| half > quarter (size step) | +0.018 | 0.027 | +0.034 | 0.003 |
| 33/33/33 > 50/25/25 @N=350 (balance) | +0.025 | 0.001 | +0.017 | 0.029 |
| gene-stratified > random 33/33/33 | −0.007 | 0.888 (ns) | +0.006 | 0.238 (ns) |


### Run XXX takeaways — size and balance matter; the "gene-stratified" trick does not

1. **Data size is the anchor — and the step is now significant.** quarter (196) 0.557/0.539 → half (385)
   0.574/0.573, paired t-test p = 0.027 (RF) / 0.003 (CatBoost). The half→full step (0.574/0.573 → 0.645/0.647) is a mixed
   size+sampling comparison (full is the un-sampled 760, not a per-gene-fractional 385), so no paired test —
   but the direction, steep and still climbing, is unchanged.
2. **At fixed size, class balance matters in the minority classes' favour.** At N=350, 33/33/33
   (0.613/0.601) beats 50/25/25 (0.588/0.584), p = 0.001 / 0.029. Same N, so the gain comes from moving
   points *out* of LOF and into GOF/NNE — **minority-class data is worth more per point.** Same mechanism
   Run XXIX found (cross-species helps by adding NNE/GOF), now shown directly at constant N.
3. **Correction: gene-stratified sampling is NOT better than random balanced.** The 3-seed run suggested
   gene-stratified 33/33/33 > random 33/33/33 (CatBoost +0.017). At 10 seeds it's null: −0.007 (RF) /
   +0.006 (CatBoost), p = 0.89 / 0.24. So preserving per-gene coverage *during balanced sampling* doesn't
   actually help beyond what random balanced sampling already does. The "better idea" was a false positive.
4. **Caveats.** The quarter→half RF step (+0.018) sits on a ±0.028 std, so it's a soft signal; CatBoost's
   +0.034 is cleaner. The balance comparison is confounded by class counts (more GOF/NNE at fixed N means
   fewer LOF) — which is the point: minority data is the binding constraint.

**Verdict: data-limited, not at a ceiling.** Size and class-balance both matter (significant), but the
highest-value direction is unchanged — more data, especially GOF/NNE minority rows — while dropping the
gene-stratified-sampling idea as a real lever.


## Run XXXI: subunit-type ablation — δ/γ/ε alone beats the full model; α and β lag

Does restricting to a single subunit class change the scores? Run α alone, β alone, and δ/γ/ε alone,
gene-aware (`cv_mode="standard"`), 3-class, RF + CatBoost, 10 seeds × 15 trials, full 82-feat set.
Delta vs full (760 = Run XXVIII reference 0.6451 / 0.6470).

### Results at a glance

| subset | n | genes | RF macro F1 | CatBoost macro F1 |
|---|---|---|---|---|
| α subunits | 517 | 9 | 0.626 | 0.616 |
| β subunits | 128 | 4 | 0.631 | 0.624 |
| δ/γ/ε subunits | 115 | 3 | **0.700** | **0.705** |


### Setup — three subunit classes, one at a time

- **Groups** (by curated `subunit` name): **α** = CHRNA1-7, CHRNA9, CHRNA10 (9 genes); **β** = CHRNB1-4
  (4 genes); **δ/γ/ε** = CHRND, CHRNE, CHRNG (3 genes). α carries the principal ACh-binding face; β and
  δ/γ/ε contribute the complementary / structural faces.
- **Protocol** matches the gene-aware runs (XXVI onward): `cv_mode="standard"` (plain StratifiedKFold),
  3-class, RF + CatBoost, 3 seeds × 15 trials, full 82-feat set. Features extracted once on all 760 rows,
  then subset by group (feature dim stays 82; the unused gene one-hot columns are constant within a subset
  and are ignored by the tree models).
- **Sizes & class split**: α 517 (LOF 265 / GOF 127 / NNE 125), β 128 (39 / 44 / 45), δ/γ/ε 115
  (73 / 24 / 18). δ/γ/ε is the smallest *and* most LOF-heavy group — its comparison is the noisiest, yet it
  scores the highest.


### Results (10 seeds × 15 trials)

| Subset | Model | n | macro F1 | std | MCC | bal acc |
|---|---|---|---|---|---|---|
| α (CHRNA1-7, 9, 10) | RF | 517 | 0.6262 | 0.0126 | 0.4507 | 0.6289 |
| α (CHRNA1-7, 9, 10) | CatBoost | 517 | 0.6158 | 0.0189 | 0.4348 | 0.6216 |
| β (CHRNB1-4) | RF | 128 | 0.6310 | 0.0407 | 0.4602 | 0.6347 |
| β (CHRNB1-4) | CatBoost | 128 | 0.6239 | 0.0149 | 0.4478 | 0.6263 |
| **δ/γ/ε (CHRND, E, G)** | RF | 115 | **0.7001** | 0.0252 | 0.5725 | 0.6936 |
| **δ/γ/ε (CHRND, E, G)** | CatBoost | 115 | **0.7046** | 0.0282 | 0.5689 | 0.7082 |

*Full (760) reference = Run XXVIII = 0.6451 (RF) / 0.6470 (CatBoost). δ/γ/ε beats α and β by +0.07–0.09 (see significance below).*

#### Conditions

| Item | Value |
|---|---|
| New thing | Split the data by subunit type (α / β / δ/γ/ε) and train separate models per group. |
| Dataset | α 517 (LOF 265/GOF 127/NNE 125) · β 128 (39/44/45) · δ/γ/ε 115 (73/24/18) |
| Genes | α = CHRNA1-7, A9, A10 (9) · β = CHRNB1-4 (4) · δ/γ/ε = CHRND/E/G (3) |
| Features | 82 (base 75 + gating 3 + allosteric 4); extracted once on all 760, then subset by group |
| Seeds | 10 (matched per-seed across groups) |
| Optuna trials | 15 per inner fold |
| CV | `standard` (5×5 StratifiedKFold, gene-aware) |
| Models | Random Forest, CatBoost |
| Metric | macro-F1 (3-class), per-seed mean ± std |

#### Significance (δ/γ/ε vs α and β, 10 seeds)

| Comparison | RF Δ | RF p | CB Δ | CB p |
|---|---|---|---|---|
| δ/γ/ε > α (paired) | +0.074 | < 0.0001 | +0.089 | < 0.0001 |
| δ/γ/ε > β (paired) | +0.069 | 0.001 | +0.081 | < 0.0001 |
| δ/γ/ε > α (Welch) | +0.074 | < 0.0001 | +0.089 | < 0.0001 |
| δ/γ/ε > β (Welch) | +0.069 | 0.0003 | +0.081 | < 0.0001 |


### Run XXXI takeaways — δ/γ/ε (muscle structural subunits) scores *above* the full model; α and β lag

1. **δ/γ/ε alone beats the full 16-gene model — on the smallest group, and even more so at 10 seeds.**
   δ/γ/ε (CHRND/E/G, 115 variants) reaches **0.700/0.705**, up from the 3-seed 0.688/0.690. That runs
   *opposite* to the data-size curve (Run XXX: fewer rows → lower score), so the gain is not "more data" —
   it's *less heterogeneity*.
2. **α and β both lag.** α (517, the biggest group) scores 0.626/0.616; β (128) scores 0.631/0.624. The
   full model's ~0.65 is being *held up* by the δ/γ/ε signal — α and β individually are each harder than
   the pooled model.
3. **The δ/γ/ε edge is statistically robust.** δ/γ/ε > α and δ/γ/ε > β are both p < 0.0001 (α) and
   p ≤ 0.001 (β), paired and Welch agree. This matches Run XXXIV's independent 10-seed significance test.
4. **Confound caveat (unchanged):** δ/γ/ε is only 3 genes, so the gene-identity feature collapses to a
   "δ vs ε vs γ" label that carries direction almost for free. The high score is real under gene-aware CV,
   but the decisive check — whether it survives leaving gene identity out — is exactly what Run XXXII ran,
   and it showed δ/γ/ε's edge is *biological*, not identity.

**Verdict:** subunit type is a real axis of heterogeneity — the muscle structural subunits (δ/γ/ε) are the
most direction-predictable slice (0.70, significant), and α/β are each harder than the pooled model.


## Run XXXII: δ/γ/ε's 0.70 is biological, not a gene-identity artifact

Run XXXI found δ/γ/ε (CHRND/E/G, 115) scores 0.700/0.705 *above* the full model — but with only 3 genes,
the `subunit_*` one-hot collapses to a "δ vs ε vs γ" label that could carry direction almost for free. This
run settles it: drop the 16 `subunit_*` columns and re-run each group, matched with/without identity.

### Results (10 seeds × 15 trials)

| subset | n | genes | RF with | RF without | CatBoost with | CatBoost without |
|---|---|---|---|---|---|---|
| full | 760 | 16 | 0.6451 | 0.6418 | 0.6470 | 0.6383 |
| α | 517 | 9 | 0.6262 | 0.6266 | 0.6158 | 0.6117 |
| β | 128 | 4 | 0.6310 | 0.6141 | 0.6239 | 0.6165 |
| **δ/γ/ε** | 115 | 3 | 0.7001 | **0.7069** | 0.7046 | **0.7094** |

#### Conditions

| Item | Value |
|---|---|
| New thing | Drop the 16 gene one-hot columns to test whether δ/γ/ε's 0.70 is a gene-identity artifact or biological. |
| Dataset | 760 variants (507 human, 173 rat, 80 mouse) |
| Classes | LOF 377 / No-net-effect 188 / GOF 195 |
| Groups | full (16 genes) · α (9) · β (4) · δ/γ/ε (3) |
| Features | 82 (with identity) vs 66 (16 `subunit_*` cols dropped) |
| Seeds | 10 (matched per-seed across with/without) |
| Optuna trials | 15 per inner fold |
| CV | `standard` 5×5 StratifiedKFold (gene-aware) |
| Models | Random Forest, CatBoost |
| Metric | macro-F1 (3-class), per-seed mean |


### Setup — drop gene identity, keep everything else

- **Question:** is δ/γ/ε's 0.70 a *gene-identity* artifact (the model reading "which of 3 genes" as a
  free direction cue) or *biological* (direction genuinely more predictable in the muscle accessory subunits)?
- **Test:** remove the 16 `subunit_*` one-hot columns (positions 1–16 of the positional group), keeping
  `position_normalized` + `species_*` + all 24 physicochemical + 3 substitution + 3 conservation + 7
  structural + 2 ESM-2 + 16 ESM-embedding features → 82 → 66 features. Re-run standard CV, matched with vs
  without identity (same seeds, same folds), per group + full.
- **Why not leave-one-gene-out:** it is degenerate for δ/γ/ε — 3 genes, and holding out CHRNG gives a
  single-class (all-LOF) test fold, while holding out CHRND leaves GOF with only 3 training rows so the
  inner StratifiedKFold(5) crashes. Dropping gene identity is the clean, decisive substitute.
- **Protocol** matches Run XXXI: `cv_mode="standard"`, 3-class, RF + CatBoost, **10 seeds × 15 trials**.
- **Sanity check:** full-with-identity reproduces the Run XXVIII 82-feat reference (RF 0.6451 / CatBoost
  0.6470), and α-with-identity reproduces Run XXXI α (0.626 / 0.616) — the matched protocol is faithful.


**Gene-identity drop — Δ = with − without identity (negative ⇒ identity was not helping):**

| Subset | n | Model | With identity | Without identity | Δ |
|---|---|---|---|---|---|
| Full (760) | 760 | RF | 0.6451 | 0.6418 | +0.003 |
| Full (760) | 760 | CatBoost | 0.6470 | 0.6383 | +0.009 |
| α (CHRNA1-7,9,10) | 517 | RF | 0.6262 | 0.6266 | −0.000 |
| α (CHRNA1-7,9,10) | 517 | CatBoost | 0.6158 | 0.6117 | +0.004 |
| β (CHRNB1-4) | 128 | RF | 0.6310 | 0.6141 | +0.017 |
| β (CHRNB1-4) | 128 | CatBoost | 0.6239 | 0.6165 | +0.007 |
| **δ/γ/ε (CHRND,E,G)** | 115 | RF | 0.7001 | **0.7069** | **−0.007** |
| **δ/γ/ε (CHRND,E,G)** | 115 | CatBoost | 0.7046 | **0.7094** | **−0.005** |

#### Significance (paired t-test over 10 seeds, with vs without identity)

| Subset | RF Δ | RF p | CB Δ | CB p |
|---|---|---|---|---|
| full | +0.003 | 0.49 | +0.009 | 0.031 |
| α | −0.000 | 0.92 | +0.004 | 0.55 |
| β | +0.017 | 0.089 | +0.007 | 0.49 |
| δ/γ/ε | −0.007 | 0.41 | −0.005 | 0.59 |

*Only the full-model CatBoost drop is significant (p = 0.031). β leans on identity most (RF Δ +0.017,
p = 0.089) but not decisively. δ/γ/ε is the only group whose sign flips negative — dropping identity does
not hurt it (ns both), so its 0.70 edge is not a gene-label lookup.*


### Run XXXII takeaways — δ/γ/ε's edge is BIOLOGICAL; the gene-identity confound is rejected

1. **δ/γ/ε survives — and even *improves* — without gene identity.** Dropping the 16 `subunit_*` columns,
   δ/γ/ε goes **RF 0.700 → 0.707 (+0.007)** and **CatBoost 0.705 → 0.709 (+0.005)**, with MCC up too
   (CatBoost 0.569 → 0.582). If the 0.70 were a "δ vs ε vs γ" lookup, removing the lookup would crash it;
   instead it rose. **The high score is carried by real features (structural/conservation/physicochemical),
   not the gene label.**
2. **Only the full model (CatBoost) and β lean on gene identity.** Full CatBoost drops 0.647 → 0.638
   (p = 0.031); β drops most per-gene (RF −0.017, p = 0.089; CatBoost −0.007, ns). α is flat
   (−0.000 / +0.004, ns). Gene identity buys ~0.01–0.02 in the pooled and β settings, but nothing in δ/γ/ε.
3. **Why δ/γ/ε goes *up* without identity:** in a 3-gene group the gene one-hot is not informative signal
   but a nuisance split the model has to route around — 3 constant columns competing with the real 66. For
   CatBoost especially, dropping them frees it to use the physicochemical/conservation signal cleanly.
4. **Read together with Run XXXI:** δ/γ/ε (the muscle *accessory* subunits — clean, well-characterised in
   congenital myasthenic syndromes) is where direction-of-effect is genuinely most predictable, and α/β are
   each harder than the pooled model. That is a *biological heterogeneity* result, not a modelling artifact.

**Verdict:** subunit type is a real axis — the muscle structural subunits are the most direction-predictable
slice, and that edge is intrinsic (it survives, and grows, when the model is forbidden from naming the gene).
Natural next step: a subunit-type-conditioned model that exploits δ/γ/ε's cleaner signal without hand-splitting
by gene.


## Run XXXIII: per-group feature ablation — δ/γ/ε leans on a *leaner* set (esm2 + gating), not "nothing"

Run XXXI found δ/γ/ε scores ~0.70 (best); Run XXXII ruled out gene identity. This run asks
*which features carry each group's signal* — by dropping one feature group at a time and testing
"does full > drop?" (one-tailed paired, matched by seed).

**Answer (10 seeds):** δ/γ/ε leans on **esm2 + gating** (a *leaner* set than α/β), and is actively
hurt by physicochemical (CatBoost) + substitution (RF). α leans on positional + conservation + gating;
β on physicochemical. But δ/γ/ε's edge (0.700 vs 0.626) is ~4× any single-feature effect, so it is
still intrinsic to the data, not carried by one engineered feature. (The 3-seed "zero load-bearing"
reading was a df=2 underpowering artifact — see takeaways.)


### Setup — drop one feature group at a time, per subunit group

- **Groups:** α (517), β (128), δ/γ/ε (115).
- **Conditions:** `full` (82 features) + 9 single-group drops (physicochemical 24, substitution 3,
  positional 20, conservation 3, structural_core 7, esm2 2, esm_embeddings 16, gating 3,
  allosteric_interaction 4).
- **Protocol:** `cv_mode="standard"` (gene-aware), 3-class, RF + CatBoost, **10 seeds** × 15 trials.
- **Readout:** Δ = full − drop; one-tailed paired t-test, matched by seed. `*` = dropping hurts
  (feature load-bearing, one-sided p < 0.05); `†` = dropping helps (feature harmful/noise, p < 0.05).


**Δ = full − drop (positive = dropping hurts). \* = load-bearing, † = harmful (both p < 0.05, one-sided).**

**alpha — full F1: RF 0.6262 / CatBoost 0.6158**

| drop | RF Δ | CatBoost Δ |
|---|---|---|
| physicochemical | −0.001 | +0.003 |
| substitution | −0.003 | −0.003 |
| positional | +0.025\* | +0.027\* |
| conservation | +0.007\* | +0.011\* |
| structural_core | +0.002 | −0.003 |
| esm2 | −0.003 | −0.004 |
| esm_embeddings | +0.002 | +0.002 |
| gating | +0.012\* | +0.012\* |
| allosteric_interaction | +0.002 | −0.005 |

**beta — full F1: RF 0.6310 / CatBoost 0.6239**

| drop | RF Δ | CatBoost Δ |
|---|---|---|
| physicochemical | +0.025\* | +0.005 |
| substitution | +0.005 | −0.001 |
| positional | −0.002 | +0.012 |
| conservation | +0.001 | +0.006 |
| structural_core | −0.005 | −0.013 |
| esm2 | +0.014 | +0.015 |
| esm_embeddings | −0.023† | −0.023† |
| gating | +0.014 | +0.009 |
| allosteric_interaction | −0.003 | −0.011 |

**delta_gamma_epsilon — full F1: RF 0.7001 / CatBoost 0.7046**

| drop | RF Δ | CatBoost Δ |
|---|---|---|
| physicochemical | −0.009 | −0.021† |
| substitution | −0.021† | +0.001 |
| positional | +0.001 | +0.006 |
| conservation | +0.005 | −0.004 |
| structural_core | +0.009 | +0.004 |
| esm2 | +0.018\* | +0.019\* |
| esm_embeddings | +0.010 | +0.011 |
| gating | +0.018\* | +0.009 |
| allosteric_interaction | +0.004 | −0.008 |

*(raw: `results/group_ablation_10seed/`)*


### Run XXXIII takeaways — δ/γ/ε leans on esm2+gating (leaner), α/β are feature-dependent

1. **δ/γ/ε leans on esm2 + gating — not "nothing".** esm2 (ESM-2 effect, 2 feats) is load-bearing for
   both models (RF +0.018 p = 0.017, CB +0.019 p = 0.036); gating (GNM) for RF (+0.018 p = 0.016).
   Physicochemical is actively *harmful* for CatBoost (−0.021 p = 0.001), substitution for RF
   (−0.021 p = 0.031). This **corrects the 3-seed "zero load-bearing" reading**, which was df=2
   underpowered (the Δ signs were right, but nothing cleared p < 0.05).
2. **α leans on positional (+0.025/+0.027), conservation (+0.007/+0.011), gating (+0.012/+0.012)** —
   all significant in both models. "Positional" = position + 16 gene one-hot + 3 species one-hot, so α
   relies partly on gene/species identity.
3. **β leans on physicochemical (RF +0.025 p = 0.011); esm_embeddings is actively harmful**
   (−0.023 both, p ≈ 0.03). The ESM-2 16-dim PCA embeddings hurt β.
4. **δ/γ/ε's edge is still not a single-feature artifact.** Its 0.700 vs α's 0.626 (gap 0.074) is ~4× any
   single-feature effect (≤ 0.021), so even dropping esm2+gating leaves δγε above α/β. The edge is
   intrinsic to the δ/γ/ε data — clean LOF+GOF labels (the per-class F1 companion, Run XXXVI), now sharpened by a specific
   esm2+gating signal rather than "diffuse robustness."


## Run XXXIV: significance — δ/γ/ε's edge is statistically robust (10 seeds, p ≤ 0.001)

Recomputes full / α / β / δ/γ/ε at 10 seeds and answers two questions with one-tailed tests:
(1) is each group above 3-class chance (1/3)? (2) is δ/γ/ε genuinely better than α / β / full?


### Setup — per-seed macro-F1 as the unit of analysis

- **Protocol:** `cv_mode="standard"`, 3-class, RF + CatBoost, **10 seeds** × 15 trials.
- **Tests:** one-sample t vs 1/3 (chance); one-tailed Welch (unpaired) and paired (seed-matched) for
  δγε > {α, β, full}.
- **Why 10 seeds:** the δγε edge from Runs XXXI–XXXIII was 1–3 seeds; this pins down whether it is real.


**1) One-sample t-test vs 3-class chance (1/3):**

| Group | Model | mean F1 ± std | p (one-tailed) |
|---|---|---|---|
| Full (760) | RF | 0.6451 ± 0.0096 | p < 0.0001 |
| Full (760) | CatBoost | 0.6470 ± 0.0129 | p < 0.0001 |
| α | RF | 0.6262 ± 0.0126 | p < 0.0001 |
| α | CatBoost | 0.6158 ± 0.0189 | p < 0.0001 |
| β | RF | 0.6310 ± 0.0407 | p < 0.0001 |
| β | CatBoost | 0.6239 ± 0.0149 | p < 0.0001 |
| δ/γ/ε | RF | 0.7001 ± 0.0252 | p < 0.0001 |
| δ/γ/ε | CatBoost | 0.7046 ± 0.0282 | p < 0.0001 |

**2) One-tailed Welch — is δ/γ/ε > α / β / full?**

| Comparison | Model | Δ mean | p (one-tailed) |
|---|---|---|---|
| δγε > α | RF | +0.0738 | p < 0.0001 |
| δγε > β | RF | +0.0691 | p = 0.0003 |
| δγε > full | RF | +0.0550 | p < 0.0001 |
| δγε > α | CatBoost | +0.0887 | p < 0.0001 |
| δγε > β | CatBoost | +0.0806 | p < 0.0001 |
| δγε > full | CatBoost | +0.0575 | p < 0.0001 |


### Run XXXIV takeaways — δ/γ/ε is the best-predicted group, and the margin is real

1. **Every group beats chance** (p < 0.0001): full 0.645/0.647, α 0.626/0.616, β 0.631/0.624,
   δ/γ/ε 0.700/0.705.
2. **δ/γ/ε > full / α / β at p ≤ 0.001** under both Welch and seed-matched paired tests
   (+0.055 to +0.089).
3. **α ≈ β** (p = 0.63–0.84), both slightly below full. Ranking locked: **δγε > full ≥ α ≈ β**.
4. The δγε result is **not a seed artifact** — it holds at 10 seeds with strong significance.


## Run XXXV: Idea A — AlphaMissense as one more feature (82 → 83): null; pathogenicity ⊥ direction

*Idea A*: add AlphaMissense pathogenicity as one more feature on top of the 82-feat baseline.
Gene-aware `cv_mode="standard"`, 3-class, RF + CatBoost, **10 seeds × 15 trials**, full 760.

**Macro F1 (82 vs 82+AM):**

| Condition | Model | Macro F1 |
| --- | --- | --- |
| 82 | random_forest | 0.6451 ± 0.0096 |
| 82 | catboost | 0.6470 ± 0.0129 |
| 82+AM | random_forest | 0.6512 ± 0.0089 |
| 82+AM | catboost | 0.6514 ± 0.0121 |

**Significance (paired t-test, 10 seeds, one-sided):** 82+AM − 82: RF +0.006 (p = 0.085) · CB +0.004
(p = 0.149) — both ns.

*(raw: `results/idea_a/`)*

### Run XXXV takeaway (Idea A) — AlphaMissense is null, re-confirming direction ⊥ pathogenicity

82 → 83 features: RF 0.6451→0.6512 (Δ +0.006, p = 0.085), CatBoost 0.6470→0.6514 (Δ +0.004, p = 0.149) —
a small positive drift that is NOT significant for either model. Re-confirms Run VIII: pathogenicity is
orthogonal to direction even as a single added feature. (The earlier 3-seed read showed RF −0.008; that sign
flip was seed variance.)

## Run XXXVI: Idea B — disease→direction label mining: tautological, not additive (+ per-class F1 companion)

*Idea B*: expand labels by mining disease associations (disease → mechanism → direction). On the
clean-label variants, 66% have NO pathology annotation, and the only ≥80%-pure pathology buckets are the
myasthenic syndromes whose *name is the direction label* — CMS (86, 86% LOF), SCCMS (35, 94% GOF), FCCMS
(6, 100% LOF). The abundant common-disease tags (nicotine dependence 41% GOF, ADNFLE 52% GOF, schizophrenia
79% LOF) are direction-mixed. Mining disease would only re-derive labels we already have, not expand
GOF/NNE. **Verdict: tautological, not additive.**

**Companion — per-class F1 per group (82-feat baseline), decomposing *why* δ/γ/ε wins:**

| Group | Model | Macro | LOF | NNE | GOF |
| --- | --- | --- | --- | --- | --- |
| α (517) | random_forest | 0.6262 | 0.7406 | 0.5965 | 0.5416 |
| α (517) | catboost | 0.6158 | 0.7263 | 0.5891 | 0.5321 |
| β (128) | random_forest | 0.6310 | 0.6384 | 0.6620 | 0.5924 |
| β (128) | catboost | 0.6239 | 0.6347 | 0.6485 | 0.5886 |
| δ/γ/ε (115) | random_forest | 0.7001 | 0.8425 | 0.5700 | 0.6877 |
| δ/γ/ε (115) | catboost | 0.7046 | 0.8308 | 0.5996 | 0.6833 |

**Significance (paired t-test, 10 seeds, one-sided):** δ/γ/ε − α: RF +0.074 (p < 0.001) · CB +0.089
(p < 0.001); δ/γ/ε − β: RF +0.069 (p = 0.001) · CB +0.081 (p < 0.001).

**Conditions:** n = 760 (α 517 / β 128 / δγε 115); features = 82 baseline; 10 seeds × 15 trials; gene-aware
`cv_mode="standard"`; models RF + CatBoost; per-seed macro-F1 is the unit.

*(raw: `results/idea_a/`)*

### Run XXXVI takeaway — δ/γ/ε's edge is LOF+GOF-specific, not all-class

Per-class: δγε LOF 0.84/0.83 + GOF 0.69/0.68, but NNE only 0.57/0.60 — the WORST NNE of the three groups
(β 0.66/0.65, α 0.60/0.59). So δγε wins because its two directional classes are clean, not because
everything separates. Refines Runs XXXII–XXXIV. (Superseded in detail by Run XXXIX's full class-wise sweep.)

## Run XXXVII: α7 resting-structure probe — complete homomeric resting model vs 7EKT (null-to-negative)

The colleague's sodium-channel precedent says a *more complete* structure improves F1. Tested by swapping
CHRNA7's structural source from 7EKT (antagonist-bound, 3.20 Å) to a new complete homomeric α7 resting model
(7KOO cryo-EM + 7RPM NMR, ligands/ions stripped, full-atom), ingested as `data/raw/structure_files/a7rest/`.
Only CHRNA7's 14 PDB features change; the other 15 genes and 68 non-structural features are identical.

Gene-aware `cv_mode="standard"`, 3-class, RF + CatBoost, **10 seeds × 15 trials**, 82-feat layout, full 760.
`naive` = all 14 PDB features from a7rest; `geom-only` = geometry/dynamics (RSA/Cβ/GNM/allosteric) from
a7rest but b_factor + DSSP held at 7EKT (the spliced model has no per-atom B-factors or SS records).


| Arm | Model | Macro F1 | LOF | NNE | GOF |
| --- | --- | --- | --- | --- | --- |
| 7EKT | random_forest | 0.6451 ± 0.0096 | 0.740 | 0.617 | 0.578 |
| 7EKT | catboost | 0.6470 ± 0.0129 | 0.736 | 0.619 | 0.586 |
| a7rest (naive) | random_forest | 0.6388 ± 0.0089 | 0.742 | 0.615 | 0.560 |
| a7rest (naive) | catboost | 0.6419 ± 0.0102 | 0.736 | 0.615 | 0.574 |
| geom-only | random_forest | 0.6404 ± 0.0085 | 0.742 | 0.612 | 0.568 |
| geom-only | catboost | 0.6487 ± 0.0114 | 0.744 | 0.618 | 0.584 |

**Δ vs 7EKT (naive, 10 seeds):**
- **random_forest** macro Δ −0.0063 (p = 0.14) · **catboost** macro Δ −0.0051 (p = 0.23)
- **GOF is the only moving class:** RF GOF Δ **−0.018 (p = 0.006)** · CatBoost GOF Δ −0.012 (p = 0.058).
  LOF (+0.001/+0.000) and NNE (−0.002/−0.004) are flat.

**Δ vs 7EKT (geom-only, 10 seeds):**
- **random_forest** macro Δ −0.0047 (p = 0.16) · **catboost** macro Δ +0.0016 (p = 0.31)
- **GOF:** RF GOF Δ −0.010 (p = 0.074) · CatBoost GOF Δ −0.002 (p = 0.40) — both now ns. Restoring
  b_factor/SS rescues ~half the naive GOF drop (geom-only − naive: RF GOF +0.008 p = 0.019, CB GOF +0.010
  p = 0.047), but geometry alone still carries no direction signal.

*(raw: `results/a7resting/comparison.json` + `results/a7resting/geomonly.json`)*


### Run XXXVII takeaways — "more complete" does not transfer; keep 7EKT

1. **Null-to-negative, not a win.** Naive swap: RF −0.0063 (p = 0.14), CatBoost −0.0051 (p = 0.23). No
   sodium-channel-style improvement; the macro score is statistically indistinguishable from 7EKT.
2. **The penalty is GOF-specific — and now significant for RF.** RF GOF drops **−0.018 (p = 0.006)**;
   CatBoost GOF −0.012 (p = 0.058, near-significant). LOF and NNE are flat. So "more complete" does not
   help, and it *actively hurts* gain-of-function prediction (the resting/apo state is *less* informative
   about activated-state variants).
3. **The naive GOF drop was ~half a metadata confound, ~half genuine geometry.** The geom-only arm (holds
   b_factor + DSSP at 7EKT, swaps only the geometry/dynamics) shrinks the GOF drop to RF −0.010 (p = 0.074)
   and CatBoost −0.002 (p = 0.40) — both ns — while geom-only − naive GOF is +0.008/+0.010 (p = 0.019/0.047).
   So the spliced model's all-zero B-factors and all-coil SS accounted for roughly half the naive penalty,
   but the remaining geometry is *still* null-to-negative, not a hidden win.
4. **Why geometry hurts: `gnm_slowmode` (global gating mode) nearly decorrelates between 7EKT and the
   resting state (corr −0.03).** The apo resting state's slowest mode is *less* informative about GOF than
   the antagonist-bound structure. The "completeness" gain is trivial (only resolves 2 residues). Keep 7EKT
   for CHRNA7.


## Run XXXVIII: functional (GO) gene embedding — a full *substitute* for identity, not a gain

Does a *functional* representation of each subunit (α/β/δγε class + mechanistic attributes:
homomeric, high-Ca2+, muscle-type, gene length) substitute for the 16 opaque gene one-hots, and do the
mechanistic attributes add signal on top of identity? (Func2Seq/Func2Prot framing.)

Gene-aware `cv_mode="standard"`, 3-class, RF + CatBoost, **10 seeds × 15 trials**, 82-feat reference
(= 75 base + gating 3 + allosteric 4). `full` = 82, `functional` = 89 (+7), `no_identity` = 66 (−16 one-hots),
`functional_no_identity` = 73 (−16 +7). Delta vs `full`.

#### Conditions

| Item | Value |
|---|---|
| New thing | Replace the 16 gene one-hots with 7 named functional attributes (α/β/δγε class + homomeric + high-Ca2+ + muscle-type + gene length). |
| Dataset | 760 variants (507 human, 173 rat, 80 mouse) |
| Classes | LOF 377 / No-net-effect 188 / GOF 195 |
| Conditions | full (82) · functional (89) · no_identity (66) · functional_no_identity (73) |
| Functional feats | 7: α/β/δγε class + homomeric + high-Ca2+ + muscle-type + gene length |
| Seeds | 10 (matched per-seed across conditions) |
| Optuna trials | 15 per inner fold |
| CV | `standard` 5×5 StratifiedKFold (gene-aware) |
| Models | Random Forest, CatBoost |
| Metric | macro-F1 (3-class), per-seed mean |


| Condition | Model | n_feat | Macro F1 | MCC | Bal acc |
| --- | --- | --- | --- | --- | --- |
| full | random_forest | 82 | 0.6451 | 0.4746 | 0.6468 |
| full | catboost | 82 | 0.6470 | 0.4782 | 0.6530 |
| functional | random_forest | 89 | 0.6523 | 0.4876 | 0.6542 |
| functional | catboost | 89 | 0.6515 | 0.4856 | 0.6571 |
| no_identity | random_forest | 66 | 0.6418 | 0.4699 | 0.6428 |
| no_identity | catboost | 66 | 0.6383 | 0.4644 | 0.6443 |
| functional_no_identity | random_forest | 73 | 0.6485 | 0.4808 | 0.6502 |
| functional_no_identity | catboost | 73 | 0.6508 | 0.4849 | 0.6576 |

*(raw: `results/functional/functional_summary.csv`)*

#### Significance (paired t-test over 10 seeds)

| Comparison | RF Δ | RF p | CB Δ | CB p |
|---|---|---|---|---|
| functional vs full (augment) | +0.007 | 0.050 | +0.004 | 0.24 |
| no_identity vs full (drop identity) | −0.003 | 0.49 | −0.009 | 0.031 |
| functional_no_identity vs no_identity (replace) | +0.007 | 0.13 | +0.012 | 0.0006 |
| functional_no_identity vs full | +0.003 | 0.42 | +0.004 | 0.35 |


### Run XXXVIII takeaways — functional encoding fully replaces identity, but adds nothing on top

1. **Augment (function on top of identity): weak.** `functional` = +0.007 (RF, p = 0.050) / +0.004
   (CatBoost, ns). The mechanistic attributes (homomeric, high-Ca2+, muscle-type, gene length) carry at
   most a sliver of direction signal beyond what the one-hots already capture — the α/β/δγε classes are
   near-reconstructible from the one-hots. RF is exactly at the p = 0.05 threshold, so at best a borderline
   positive.
2. **Dropping identity is surprisingly cheap.** `no_identity` = −0.003 (RF, ns) / −0.009 (CatBoost,
   p = 0.031) — *not* the −0.13 gene-agnostic gap from Run XXVI. Under gene-leaky `cv_mode="standard"`,
   the model reconstructs gene identity from gene-correlated features (structural PDB chain, ESM-2,
   position-normalized, gene length) anyway. The 0.50 → 0.63 jump was a **CV-protocol effect, not a
   one-hot-column effect** (consistent with Run XXXII).
3. **Function *fully* substitutes for identity.** `functional_no_identity` recovers **100%+** of the small
   identity drop — it is statistically indistinguishable from `full` (+0.003 / +0.004, ns), and for CatBoost
   the recovery over `no_identity` is significant (**+0.012, p = 0.0006**). The 3-seed "~44–80% substitute"
   was an underestimate; at 10 seeds the 7 mechanistic features *completely* stand in for the 16 one-hots.
4. **Verdict: a compression, not a gain.** The functional encoding is a legitimate *alternative* to the
   16 one-hots — more interpretable (named mechanistic classes vs opaque IDs) at zero cost — but it adds no
   independent signal on top. Direction-of-effect is gene-conditioned (Run XXVI); static functional classes
   can *encode* that conditioning, they just can't *improve* it. 12th hypothesis that doesn't move the needle.


## Run XXXIX: class-wise F1 + "is the model cheating?" — per-class decomposition, baselines, and the δ/γ/ε always-LOF test

Three questions in one sweep: (1) per-class F1 (LOF / No-net-effect / GOF) for every model, on the full data
and on each subunit group alone; (2) baselines — majority-class ("always predict the mode"), uniform-random
(chance), and stratified-random (priors); (3) the δ/γ/ε cheating test — since LOF is 63.5% of δγε, how much
macro F1 does a real model earn over a classifier that *always predicts LOF*?

Gene-aware `cv_mode="standard"`, 3-class, all 11 models, 5 outer × 15 Optuna trials, **10 seeds**, full 82-feat
set. Baselines are analytic (DummyClassifier over the whole subset). Full table:
`results/classwise_f1/classwise_f1_summary.csv`.


**all (n=760)** — rows 0–2 are baselines (majority / uniform / stratified):

| Model | Macro F1 | LOF | NNE | GOF |
| --- | --- | --- | --- | --- |
| majority_class | 0.221 | 0.663 | 0.000 | 0.000 |
| uniform_random | 0.324 | 0.398 | 0.284 | 0.290 |
| stratified_random | 0.335 | 0.498 | 0.249 | 0.258 |
| logistic_regression | 0.587 | 0.659 | 0.604 | 0.499 |
| svm_rbf | 0.595 | 0.702 | 0.543 | 0.541 |
| random_forest | 0.645 | 0.740 | 0.617 | 0.578 |
| lightgbm | 0.641 | 0.730 | 0.618 | 0.576 |
| svm_linear | 0.602 | 0.676 | 0.606 | 0.525 |
| knn | 0.573 | 0.696 | 0.480 | 0.544 |
| xgboost | 0.643 | 0.759 | 0.613 | 0.558 |
| catboost | 0.647 | 0.736 | 0.619 | 0.586 |
| mlp | 0.597 | 0.697 | 0.564 | 0.529 |
| gaussian_nb | 0.513 | 0.562 | 0.510 | 0.466 |
| ordinal | 0.559 | 0.694 | 0.467 | 0.518 |

**α (n=517)** — rows 0–2 are baselines (majority / uniform / stratified):

| Model | Macro F1 | LOF | NNE | GOF |
| --- | --- | --- | --- | --- |
| majority_class | 0.226 | 0.678 | 0.000 | 0.000 |
| uniform_random | 0.321 | 0.403 | 0.278 | 0.281 |
| stratified_random | 0.331 | 0.512 | 0.236 | 0.245 |
| logistic_regression | 0.559 | 0.668 | 0.557 | 0.452 |
| svm_rbf | 0.565 | 0.697 | 0.512 | 0.484 |
| random_forest | 0.626 | 0.741 | 0.596 | 0.542 |
| lightgbm | 0.611 | 0.716 | 0.588 | 0.530 |
| svm_linear | 0.569 | 0.676 | 0.553 | 0.476 |
| knn | 0.539 | 0.680 | 0.440 | 0.498 |
| xgboost | 0.603 | 0.752 | 0.582 | 0.475 |
| catboost | 0.616 | 0.726 | 0.589 | 0.532 |
| mlp | 0.565 | 0.705 | 0.528 | 0.463 |
| gaussian_nb | 0.518 | 0.619 | 0.513 | 0.422 |
| ordinal | 0.535 | 0.691 | 0.441 | 0.474 |

**β (n=128)** — rows 0–2 are baselines (majority / uniform / stratified):

| Model | Macro F1 | LOF | NNE | GOF |
| --- | --- | --- | --- | --- |
| majority_class | 0.173 | 0.000 | 0.520 | 0.000 |
| uniform_random | 0.331 | 0.314 | 0.336 | 0.343 |
| stratified_random | 0.333 | 0.306 | 0.350 | 0.344 |
| logistic_regression | 0.624 | 0.650 | 0.644 | 0.578 |
| svm_rbf | 0.582 | 0.643 | 0.571 | 0.532 |
| random_forest | 0.631 | 0.638 | 0.662 | 0.592 |
| lightgbm | 0.606 | 0.579 | 0.641 | 0.596 |
| svm_linear | 0.611 | 0.653 | 0.627 | 0.552 |
| knn | 0.592 | 0.621 | 0.635 | 0.519 |
| xgboost | 0.631 | 0.622 | 0.665 | 0.604 |
| catboost | 0.624 | 0.635 | 0.648 | 0.589 |
| mlp | 0.572 | 0.618 | 0.572 | 0.525 |
| gaussian_nb | 0.577 | 0.632 | 0.594 | 0.505 |
| ordinal | 0.581 | 0.600 | 0.575 | 0.568 |

**δ/γ/ε (n=115)** — rows 0–2 are baselines (majority / uniform / stratified):

| Model | Macro F1 | LOF | NNE | GOF |
| --- | --- | --- | --- | --- |
| majority_class | 0.259 | 0.777 | 0.000 | 0.000 |
| uniform_random | 0.298 | 0.435 | 0.203 | 0.256 |
| stratified_random | 0.336 | 0.634 | 0.169 | 0.207 |
| logistic_regression | 0.664 | 0.770 | 0.614 | 0.608 |
| svm_rbf | 0.696 | 0.824 | 0.613 | 0.653 |
| random_forest | 0.700 | 0.842 | 0.570 | 0.688 |
| lightgbm | 0.687 | 0.817 | 0.575 | 0.668 |
| svm_linear | 0.683 | 0.788 | 0.642 | 0.619 |
| knn | 0.677 | 0.822 | 0.577 | 0.631 |
| xgboost | 0.668 | 0.830 | 0.517 | 0.656 |
| catboost | 0.705 | 0.831 | 0.600 | 0.683 |
| mlp | 0.682 | 0.791 | 0.636 | 0.618 |
| gaussian_nb | 0.594 | 0.654 | 0.614 | 0.515 |
| ordinal | 0.658 | 0.771 | 0.627 | 0.575 |

*(raw: `results/classwise_f1/classwise_f1_summary.csv`)*

### How to read the baselines (rows 0–2 of each table)

- **majority_class** — "always predict the most common effect" (DummyClassifier `most_frequent`). It never
  predicts the other two classes, so their columns are 0.000 *by construction*; its macro F1 is
  (majority-F1 + 0 + 0)/3. For α and δ/γ/ε (LOF-majority) this is the "always-predict-LOF" floor; for β
  (NNE-majority) it is "always-predict-NNE".
- **uniform_random** — "predict each of the 3 classes with equal 1/3 chance" (the chance floor). Because it
  guesses NNE and GOF a third of the time, those columns are non-zero (~0.28–0.34), unlike majority.
- **stratified_random** — "predict each class with probability = its frequency" (chance, but class-prior-aware).
  Slightly better than uniform on the majority class, worse on the minority; net macro ≈ chance.

These are *analytic* floors (no CV, no learning) — they answer "is the model doing better than a dumb rule?".
The δ/γ/ε table is the explicit "is 0.70 just predicting LOF?" check: always-LOF reaches only 0.259, so the
model's ~0.70 is genuine 3-way discrimination.


### Run XXXIX takeaways — δ/γ/ε is NOT cheating; every model reads all three directions

1. **δ/γ/ε's 0.70 is not "predict LOF".** LOF is 63.5% of δγε, so always-LOF reaches macro F1 0.259
   (LOF 0.777 / NNE 0.000 / GOF 0.000). Every model crushes that by **+0.34 to +0.45 macro F1**, with
   **GOF-F1 0.52–0.69** and **NNE-F1 0.52–0.64** — classes an always-LOF model scores 0.000 on. The edge is
   genuine 3-way discrimination, not a majority-class crutch. (Independently confirms Run XXXII: δγε even
   *improves* when gene identity is dropped.)
2. **Best δγε model is CatBoost (0.705) / RF (0.700) — the same pair as the headline.** SVM-RBF (0.696) and
   LightGBM (0.687) follow. XGBoost's 3-seed 0.719 was seed variance (its 10-seed std is 0.054 vs ~0.025 for
   the others); at 10 seeds it lands at 0.668.
3. **β is the only balanced group (LOF 39 / NNE 45 / GOF 44)** and is the hardest: majority (NNE) is only 35%,
   so its majority baseline is the lowest floor (0.173), yet models reach 0.57–0.63 with all three classes
   0.50–0.67. α and full are LOF-majority (~50%) like δγε but at lower purity (0.51/0.50 vs 0.63).
4. **Every model beats the stratified-random baseline everywhere** (best macro 0.336), and GaussianNB is the
   consistent laggard (0.51–0.59), as expected for a small tabular dataset.


## Gene identity matrix (16 human nAChR subunits)

Pairwise **BLOSUM62 global-alignment % identity** between the 16 canonical human RefSeq isoforms (diagonal = self = 100%). Computed by `scripts/gene_identity_matrix.py`; saved to `data/raw/reference_sequences/mapping/gene_identity_matrix.csv`.

For reference, **cross-species** identity (human ↔ mouse/rat, *same* gene) is **81–94%** — i.e. the genes are far more different from *each other* (27–63%) than the same gene is across species.

| gene | CHRNA1 | CHRNA2 | CHRNA3 | CHRNA4 | CHRNA5 | CHRNA6 | CHRNA7 | CHRNA9 | CHRNA10 | CHRNB1 | CHRNB2 | CHRNB3 | CHRNB4 | CHRND | CHRNE | CHRNG |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| CHRNA1 | 100.0 | 42.2 | 45.9 | 37.5 | 43.2 | 42.7 | 34.8 | 34.3 | 34.3 | 34.9 | 39.2 | 42.2 | 40.0 | 34.4 | 29.0 | 30.4 |
| CHRNA2 | 42.2 | 100.0 | 53.8 | 54.9 | 47.2 | 49.2 | 36.8 | 33.7 | 33.8 | 36.8 | 46.0 | 48.0 | 45.0 | 33.5 | 32.5 | 33.0 |
| CHRNA3 | 45.9 | 53.8 | 100.0 | 47.1 | 41.4 | 63.5 | 36.3 | 32.8 | 34.8 | 35.3 | 42.9 | 43.6 | 43.7 | 36.8 | 34.1 | 31.1 |
| CHRNA4 | 37.5 | 54.9 | 47.1 | 100.0 | 38.7 | 45.5 | 33.1 | 29.2 | 29.3 | 33.4 | 42.0 | 40.2 | 40.3 | 31.1 | 28.5 | 30.9 |
| CHRNA5 | 43.2 | 47.2 | 41.4 | 38.7 | 100.0 | 42.5 | 30.9 | 32.1 | 33.6 | 33.1 | 38.2 | 62.9 | 39.7 | 32.5 | 30.8 | 30.2 |
| CHRNA6 | 42.7 | 49.2 | 63.5 | 45.5 | 42.5 | 100.0 | 34.6 | 33.9 | 33.7 | 34.5 | 42.7 | 44.6 | 42.7 | 36.4 | 32.5 | 33.3 |
| CHRNA7 | 34.8 | 36.8 | 36.3 | 33.1 | 30.9 | 34.6 | 100.0 | 34.6 | 36.8 | 30.8 | 34.5 | 31.6 | 37.1 | 30.2 | 28.7 | 30.9 |
| CHRNA9 | 34.3 | 33.6 | 32.8 | 29.2 | 32.1 | 33.9 | 34.6 | 100.0 | 51.9 | 28.5 | 30.9 | 32.3 | 31.1 | 28.9 | 26.8 | 28.0 |
| CHRNA10 | 34.3 | 33.8 | 34.8 | 29.3 | 33.6 | 33.7 | 36.8 | 51.9 | 100.0 | 30.6 | 34.0 | 33.5 | 34.1 | 29.7 | 30.4 | 31.1 |
| CHRNB1 | 34.9 | 36.8 | 35.3 | 33.4 | 33.1 | 34.5 | 30.8 | 28.5 | 30.6 | 100.0 | 41.3 | 35.4 | 39.4 | 39.1 | 39.2 | 40.7 |
| CHRNB2 | 39.2 | 46.0 | 42.9 | 42.0 | 38.2 | 42.7 | 34.5 | 30.9 | 34.0 | 41.3 | 100.0 | 38.4 | 60.8 | 40.0 | 35.6 | 39.1 |
| CHRNB3 | 42.2 | 47.9 | 43.6 | 40.2 | 62.9 | 44.6 | 31.6 | 32.3 | 33.5 | 35.4 | 38.4 | 100.0 | 40.6 | 32.7 | 31.6 | 30.7 |
| CHRNB4 | 40.0 | 45.0 | 43.7 | 40.3 | 39.7 | 42.7 | 37.1 | 31.1 | 34.1 | 39.4 | 60.8 | 40.6 | 100.0 | 38.8 | 35.3 | 37.9 |
| CHRND | 34.4 | 33.5 | 36.8 | 31.1 | 32.5 | 36.4 | 30.2 | 28.9 | 29.7 | 39.1 | 40.0 | 32.7 | 38.8 | 100.0 | 44.7 | 46.9 |
| CHRNE | 29.0 | 31.8 | 34.1 | 28.5 | 30.8 | 32.5 | 28.7 | 26.8 | 30.4 | 39.2 | 35.6 | 31.6 | 35.3 | 44.7 | 100.0 | 50.1 |
| CHRNG | 30.4 | 33.0 | 31.1 | 30.9 | 30.2 | 33.3 | 30.9 | 28.0 | 31.1 | 40.7 | 39.1 | 30.7 | 38.0 | 46.9 | 50.1 | 100.0 |

## Run XL: single-frame mapping (all 16 genes → CHRNA2)

**Setup** — same pipeline, same seeds, same feature set; the only change is the coordinate system. `cv_mode="standard"` (gene-aware headline), 10 seeds, 5 outer folds, 5 Optuna trials, dropped extractors = ['esm2', 'esm_embeddings'] (for speed, identically on both arms).

**Arms:** (1) *baseline* = 16 genes, real gene identity + human-numbered positions; (2) *single-frame* = every variant remapped onto CHRNA2 (the most central subunit) — gene identity collapsed to one constant column, positions rewritten to the CHRNA2-aligned residue (`scripts/map_genes_to_single_frame.py`).

| model | baseline | single-frame | delta |
|---|---|---|---|
| random_forest | 0.6427 | 0.5797 | -0.0629 |
| catboost | 0.6328 | 0.5781 | -0.0546 |

**Takeaway** — collapsing all 16 genes onto one reference frame *costs* ~0.05–0.06 macro-F1. It is not a free lunch: the mapping is lossy (~45% of variants land on a non-homologous CHRNA2 residue), and the gene identity the model was relying on is gone. The remaining ~0.58 is carried by species + amino-acid features, not by a valid cross-gene position signal.